In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import gc
from tqdm import tqdm
from torch.utils.data import DataLoader, Subset, TensorDataset
from bioplnn.models import SpatiallyEmbeddedClassifier, SpatiallyEmbeddedRNN
from bioplnn.utils import (
    initialize_dataloader,
)
from scipy.ndimage import center_of_mass

from collections import deque
# from umap import UMAP

from sklearn.linear_model import LogisticRegression

from sklearn.decomposition import PCA
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401
from matplotlib.cm import get_cmap
import statsmodels.api as sm

import pickle
import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection
plt.rcParams['figure.dpi'] = 300


checkpoint_path = "./train/checkpoints/"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

maze_data_path = "./data/mazes/"
checkpoint_path = "./train/checkpoints/"

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self, in_channels=4, num_classes=2, dropout=0.3):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, 64, kernel_size=5, padding=2)
        self.conv2 = nn.Conv2d(64, 64, kernel_size=5, padding=2)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=5, padding=2)
        self.pool  = nn.MaxPool2d(2, 2)
        self.relu  = nn.ReLU(inplace=True)
        self.dropout = nn.Dropout(dropout)
        self.gap   = nn.AdaptiveAvgPool2d(1)     # <- NEW
        self.fc1   = nn.Linear(128, 512)          # <- 64 channels only
        self.fc2   = nn.Linear(512, num_classes)

    def forward(self, x):
        x = self.pool(self.relu(self.conv1(x)))
        x = self.pool(self.relu(self.conv2(x)))
        x = self.pool(self.relu(self.conv3(x)))
        x = self.gap(x)                          # [B, 64, 1, 1]
        x = torch.flatten(x, 1)                  # [B, 64]
        x = self.dropout(self.relu(self.fc1(x)))
        return self.fc2(x)

def prepare_rnn_weights(state_dict):
    """Remove 'rnn.' prefix from all keys in the state dict.
        Also remove any key starting with readout"""
    new_state_dict = {}
    for key, value in state_dict.items():
        if key.startswith('rnn.'):
            new_key = key[4:]  # Remove 'rnn.' prefix
            new_state_dict[new_key] = value
        elif not key.startswith('readout'):
            new_state_dict[key] = value
    return new_state_dict
    
    
def load_model_and_config(wandb_name, checkpoint_path):
    """Load model configuration and instantiate models."""
    try:
        full_cfg = pickle.load(open(checkpoint_path + f"{wandb_name}.pkl", "rb"))
        model_cfg, num_steps = full_cfg["model_config"], full_cfg["num_steps"]
    except:
        model_cfg = pickle.load(open(checkpoint_path + f"{wandb_name}.pkl", "rb"))
        num_steps = 20
        
    try:
        if full_cfg["model_type"] == "cnn":
            model = SimpleCNN(in_channels=model_cfg["in_channels"], num_classes=model_cfg["num_classes"])
            classifier = SimpleCNN(in_channels=model_cfg["in_channels"], num_classes=model_cfg["num_classes"])
        else:
            model = SpatiallyEmbeddedRNN(**model_cfg["rnn_kwargs"])
            classifier = SpatiallyEmbeddedClassifier(**model_cfg)
    except:
        model = SpatiallyEmbeddedRNN(**model_cfg["rnn_kwargs"])
        classifier = SpatiallyEmbeddedClassifier(**model_cfg)

        # ——— Load checkpoint & weights ———
    try:
        state_dict = torch.load(checkpoint_path + f"{wandb_name}_{checkpoint}.pth", map_location=torch.device('cpu'))
    except:
        state_dict = torch.load(checkpoint_path + f"{wandb_name}/{checkpoint}.pth", map_location=torch.device('cpu'))

    try:
        state_dict = state_dict["model_state"]
    except:
        pass

    model.load_state_dict(prepare_rnn_weights(state_dict))
    classifier.load_state_dict(state_dict)

    return model, classifier, num_steps, full_cfg, state_dict

In [ ]:
def random_two_one_squares(x: torch.Tensor, *, tol: float = 0.0, gen=None):
    """
    Find two non-overlapping 2x2 squares of 1s in x and return:
      - dot_input: a zero-like mask with the two 2x2 squares set to 1
      - squares: list of length 2, each an (4, 2) tensor of (row, col) indices
    """
    if x.dim() != 2:
        raise ValueError("x must be a 2D tensor (H, W).")

    # cells equal to 1 (optionally within tolerance for floats)
    mask = (x == 1) if tol == 0 else ((x >= 1 - tol) & (x <= 1 + tol))

    H, W = mask.shape
    if H < 2 or W < 2:
        raise ValueError("x must be at least 2x2 to contain a 2x2 square.")

    # True where the 2x2 window (top-left at (i,j)) is all ones
    tl = mask[:-1, :-1]
    tr = mask[:-1,  1:]
    bl = mask[ 1:, :-1]
    br = mask[ 1:,  1:]
    top_left_ok = tl & tr & bl & br  # shape (H-1, W-1)

    candidates = torch.nonzero(top_left_ok, as_tuple=False)  # [(i,j) top-lefts], shape [K, 2]
    K = candidates.size(0)
    if K < 2:
        raise ValueError("Need at least two 2x2 squares of ones (non-overlapping).")

    # Randomize candidate order
    if gen is None:
        perm = torch.randperm(K, device=candidates.device)
    else:
        perm = torch.randperm(K, generator=gen, device=candidates.device)
    candidates = candidates[perm]

    # Helper to test overlap between two 2x2 squares given their top-lefts (i1,j1) and (i2,j2)
    # They overlap iff |i1 - i2| < 2 AND |j1 - j2| < 2 (they share at least one cell).
    def non_overlapping(a, b):
        return not (abs(int(a[0]) - int(b[0])) < 2 and abs(int(a[1]) - int(b[1])) < 2)

    # Pick first, then find a non-overlapping second
    first = candidates[0]
    second = None
    for c in candidates[1:]:
        if non_overlapping(first, c):
            second = c
            break

    if second is None:
        raise ValueError("Found multiple 2x2 squares, but none are non-overlapping. Try relaxing tol or input.")

    # Build output mask and explicit indices
    dot_input = torch.zeros_like(x, dtype=x.dtype)
    squares = []
    for tlrc in (first, second):
        i, j = int(tlrc[0]), int(tlrc[1])
        coords = torch.tensor([[i, j],
                               [i, j+1],
                               [i+1, j],
                               [i+1, j+1]], device=x.device, dtype=torch.long)
        squares.append(coords)
        dot_input[i, j] = 1
        dot_input[i, j+1] = 1
        dot_input[i+1, j] = 1
        dot_input[i+1, j+1] = 1

    return dot_input, squares

In [ ]:
def get_activations(model, classifier, num_samples, loader, decision_step, num_steps=20, dim=48, device=None):
    if device is None:
        device = next(model.parameters()).device

    # choose a sorted subset of indices
    all_indices = np.arange(len(loader.dataset))
    subset_indices = np.random.choice(all_indices, size=num_samples, replace=False)
    subset_indices = np.sort(subset_indices)

    # wrap dataset
    subset_dataset = Subset(loader.dataset, subset_indices)

    # mirror original loader settings where possible
    dataloader = DataLoader(
        subset_dataset,
        batch_size=loader.batch_size,
        shuffle=False,  # keep consistent ordering for preallocation fill
        num_workers=getattr(loader, "num_workers", 0),
        pin_memory=getattr(loader, "pin_memory", False),
        drop_last=False
    )

    # preallocate
    e_a = np.zeros((num_samples, num_steps, 8, dim, dim), dtype=np.float32)
    i_a = None #np.zeros((num_samples, num_steps, 4, dim, dim), dtype=np.float32)
    o_a = None #np.zeros((num_samples, num_steps, 8, dim, dim), dtype=np.float32)
    decisions = np.zeros((num_samples,), dtype=np.int64)

    model.eval()
    classifier.eval()
    classifier.to(device)

    final_labels = []
    final_inputs = torch.zeros((num_samples, 30, 1, 128, 128))

    sample_idx = 0
    with torch.no_grad():
        for batch_inputs, labels in tqdm(dataloader):
            bsz = batch_inputs.shape[0]
            try:
                final_inputs[sample_idx:sample_idx + bsz] = batch_inputs
            except:
                print("You hardcoded final_inputs shape dingus")
                raise Exception("You hardcoded final_inputs shape dingus")
            
            batch_inputs = batch_inputs.to(device, non_blocking=True)
            # core model forward once
            batch_output_states, batch_neuron_states, batch_feedback_states = model(
                batch_inputs, num_steps=num_steps
            )

            # --- collect O ---
            # your original code: batch_output_states[0] (shape: [B, T, 8, H, W])
            o = batch_output_states[0].detach().cpu().numpy()  # (B, T, 8, dim, dim)

            # --- collect E and I via query_neuron_states ---
            # your original code: query_neuron_states(states, layer_idx, 0 for 'e', 1 for 'i')
            e = model.query_neuron_states(batch_neuron_states, 0, 0).detach().cpu().numpy()  # (B, T, 8, dim, dim)
            i = model.query_neuron_states(batch_neuron_states, 0, 1).detach().cpu().numpy() # (B, T, 4, dim, dim)

            # --- decisions in the same pass ---
            # your original code called classifier(batch_inputs, num_steps=decision_step) and argmax over last dim
            pred = classifier(batch_inputs, num_steps=decision_step)            # (B, num_classes) or similar
            dec = torch.argmax(pred, dim=-1).detach().cpu().numpy()            # (B,)

            # write into preallocated buffers
            sl = slice(sample_idx, sample_idx + bsz)
            e_a[sl, :, :, :, :] = e
            # i_a[sl, :, :, :, :] = i
            # o_a[sl, :, :, :, :] = o
            decisions[sl] = dec

            final_labels.extend(labels.flatten().tolist())

            sample_idx += bsz
    
    return e_a, i_a, o_a, decisions, final_inputs, final_labels

def get_decisions(classifier, dataloader, decision_step, device=None):
    if device is None:
        device = next(classifier.parameters()).device

    num_samples = len(dataloader.dataset)
    decisions = np.zeros((num_samples,), dtype=np.int64)

    classifier.eval()
    classifier.to(device)

    sample_idx = 0
    with torch.no_grad():
        for batch_inputs, _labels in tqdm(dataloader):
            bsz = batch_inputs.shape[0]
            batch_inputs = batch_inputs.to(device, non_blocking=True)
            # --- decisions in the same pass ---
            # your original code called classifier(batch_inputs, num_steps=decision_step) and argmax over last dim
            pred = classifier(batch_inputs, num_steps=decision_step)            # (B, num_classes) or similar
            dec = torch.argmax(pred, dim=-1).detach().cpu().numpy()            # (B,)

            # write into preallocated buffers
            sl = slice(sample_idx, sample_idx + bsz)
            decisions[sl] = dec
            sample_idx += bsz

    return decisions

In [ ]:
def get_start_coordinates(maze, start_goal):
    H, W = maze.shape

    # 1) Flood-fill start_goal into two blocks
    visited = np.zeros_like(start_goal, bool)
    blocks = []
    for (i,j), v in np.ndenumerate(start_goal):
        if v and not visited[i,j]:
            comp, q = [], deque([(i,j)])
            visited[i,j] = True
            while q:
                x,y = q.popleft()
                comp.append((x,y))
                for dx,dy in [(1,0),(-1,0),(0,1),(0,-1)]:
                    nx, ny = x+dx, y+dy
                    if (0<=nx<H and 0<=ny<W and start_goal[nx,ny]
                        and not visited[nx,ny]):
                        visited[nx,ny] = True
                        q.append((nx,ny))
            blocks.append(comp)
    
    return blocks
    
def get_dist(maze, start_goal):
    """
    Returns an (H,W) uint8 mask marking corridor cells on any shortest path
    between the two 2×2 input blocks. If the blocks are disconnected, returns
    all zeros.
    """
    H, W = maze.shape

    blocks = get_start_coordinates(maze, start_goal)
    
    if len(blocks) != 2:
        return 0

    # 2) BFS distances from each block
    dist0 = bfs_distance(maze, blocks[0])
    dist1 = bfs_distance(maze, blocks[1])

    # 3) Minimal distance between blocks
    d_min = min(dist0[i,j] for (i,j) in blocks[1])
    return d_min


def get_maze_variations(all_samples, idx=3, num_samples=1000):
    input = all_samples[idx][0]

    inputs = []
    np.random.seed(0)
    for _ in tqdm(range(num_samples)):
        new_input = input[0].clone()
        new_input[-1] = random_two_one_squares(input[0,0])[0]
        inputs.append(new_input)
    inputs = torch.stack(inputs)
    return inputs

def plot_time_to_solution():
    inputs = get_maze_variations()

    close = inputs[19]
    mid = inputs[47]
    far = inputs[12]
    negative = inputs[49]

    classifier.to(device)
    logits_list = []
    for num_steps in range(1, 30):
        logits = classifier(mid.unsqueeze(0).to(device), num_steps=num_steps)
        logits_list.append(logits[0][1].detach().cpu().item()-logits[0][0].detach().cpu().item())
    
    plt.plot(logits_list)

In [ ]:
class PCTrajectoryPlotter:
    """
    Fit PCA on (N, T, D) activations and plot trajectories using the fitted PCA.

    Usage:
        p = PCTrajectoryPlotter(max_components=10, random_state=0)
        p.fit(activations_train, decisions_train, balance_for_fit=False)
        p.plot(activations_val, decisions_val, n_components=2, ...)
        p.plot(activations_val, decisions_val, n_components=3, ...)
        p.plot(activations_val, decisions_val, n_components=5, ...)
    Key behaviors:
      - PCA is fit ONCE via `.fit(...)` with max_components. Later `.plot(...)` calls reuse it and can select n_components <= max_components.
      - You can pass different activations/decisions to `.plot(...)`.
      - Class color mapping is fixed based on classes seen at fit-time.
      - Overlay exemplars can be restricted to a subset of classes via `overlay_classes`.
      - plot_all_endpoints scatters the ENDPOINT of *all* trials, regardless of other settings.
    """

    def __init__(self, max_components=None, n_components=None, random_state=0):
        # Backward compatibility: n_components is an alias for max_components if max_components is not given
        if max_components is None and n_components is not None:
            max_components = n_components
        self.max_components = max_components
        self.random_state = random_state

        # For backward compatibility, keep n_components as a property
        self.n_components = max_components

        # PCA will be fit with max_components (or all if None)
        self.pca = None
        self.fitted_ = False

        # Fitted metadata
        self.classes_ = None            # sorted unique class values (as seen in decisions at fit time)
        self.class_to_index_ = None     # mapping from class value -> integer index
        self.color_map_ = None          # mapping from class index -> rgba
        self.cmap_name = 'viridis'
        self.fit_collapsed_ = False
        self._fit_D = None              # Number of features at fit time

    # ----------------------- Core utilities -----------------------

    @staticmethod
    def _check_inputs(A, y):
        A = np.asarray(A)
        y = np.asarray(y)
        assert A.ndim == 3, f"activations must be (N,T,D); got {A.shape}"
        N, T, D = A.shape
        assert y.shape[0] == N, f"decisions length {y.shape[0]} must match N {N}"
        return A, y, N, T, D

    @staticmethod
    def _balance_indices(y_num, rng):
        counts = np.bincount(y_num)
        k_bal = counts[counts > 0].min()
        uniq = np.unique(y_num)
        keep = np.concatenate([
            rng.choice(np.where(y_num == c)[0], k_bal, replace=False) for c in uniq
        ])
        keep.sort()
        return keep

    def _build_color_map(self, uniq_indices):
        cmap = plt.get_cmap(self.cmap_name)
        if len(uniq_indices) <= 1:
            return {uniq_indices[0]: cmap(0.5)}
        return {c: cmap(i / max(1, len(uniq_indices) - 1)) for i, c in enumerate(uniq_indices)}

    def _resolve_overlay_class_indices(self, overlay_classes):
        """Convert overlay_classes (values) to internal indices used at *plot time*.
           Any classes not seen at fit time are ignored silently."""
        if overlay_classes is None:
            # All fit-time classes
            return set(range(len(self.classes_)))
        out = set()
        for c in overlay_classes:
            if c in self.class_to_index_:
                out.add(self.class_to_index_[c])
        return out

    # ----------------------- Fit / Transform -----------------------

    def fit(self, activations, decisions, *, balance_for_fit=False, collapse_to_class_means=False):
        """
        Fit PCA on (N,T,D) activations.
          - balance_for_fit=True balances class counts before PCA.
          - collapse_to_class_means=True fits PCA on per-class mean trajectories only.
        """
        from sklearn.decomposition import PCA

        rng = np.random.default_rng(self.random_state)
        A, y, N, T, D = self._check_inputs(activations, decisions)
        self._fit_D = D

        # establish classes at fit time
        classes, y_num = np.unique(y, return_inverse=True)
        self.classes_ = classes
        self.class_to_index_ = {c: i for i, c in enumerate(classes)}

        if collapse_to_class_means:
            # one mean trajectory per class
            A_means = []
            for c in np.unique(y_num):
                idx = np.where(y_num == c)[0]
                A_means.append(A[idx].mean(axis=0))  # (T,D)
            A_fit = np.stack(A_means, axis=0)  # (C,T,D)
            X_fit = A_fit.reshape(-1, D)
            self.fit_collapsed_ = True
        else:
            if balance_for_fit:
                counts = np.bincount(y_num)
                k_bal = counts[counts > 0].min()
                keep = np.concatenate([
                    rng.choice(np.where(y_num == c)[0], k_bal, replace=False) for c in np.unique(y_num)
                ])
                keep.sort()
                A = A[keep]
            X_fit = A.reshape(-1, D)
            self.fit_collapsed_ = False

        # If max_components is None, use all possible components (D)
        n_components = self.max_components if self.max_components is not None else min(X_fit.shape[0], D)
        self.pca = PCA(n_components=n_components, random_state=self.random_state)
        self.pca.fit(X_fit)
        uniq_indices = np.arange(len(classes))
        self.color_map_ = self._build_color_map(uniq_indices)
        self.fitted_ = True
        return self

    def transform(self, activations, n_components=None):
        """
        Project (N,T,D) activations into PC space using fitted PCA.
        n_components: number of PCs to return (<= max_components fit at .fit time).
        """
        assert self.fitted_, "Call .fit(...) first."
        A = np.asarray(activations)
        assert A.ndim == 3
        N, T, D = A.shape
        if D != self._fit_D:
            raise ValueError(f"Input feature dimension {D} does not match fit-time dimension {self._fit_D}")
        pcs_all = self.pca.transform(A.reshape(-1, D)).reshape(N, T, -1)
        # Determine how many components to return
        if n_components is None:
            n_components = self.max_components if self.max_components is not None else pcs_all.shape[-1]
        if n_components > pcs_all.shape[-1]:
            raise ValueError(f"Requested n_components={n_components} > max_components fit ({pcs_all.shape[-1]})")
        return pcs_all[..., :n_components]

    # ----------------------- Plot helpers -----------------------

    def _pick_indices_per_class(self, y_num, pcs_trials, k, mode, overlay_class_indices, rng):
        """Pick exemplar indices per class using ENDPOINT distance (closest/farthest/random)."""
        uniq = np.unique(y_num)
        chosen = []
        for c in uniq:
            if c not in overlay_class_indices:
                continue
            idx = np.where(y_num == c)[0]
            if k <= 0 or k >= len(idx):
                chosen.extend(idx); continue
            pcs_c = pcs_trials[idx]  # [Nc, T, C]
            mean_end = pcs_c[:, -1, :].mean(axis=0)
            d_end = ((pcs_c[:, -1, :] - mean_end) ** 2).sum(axis=1)
            if mode == "closest":
                sel_local = np.argsort(d_end)[:k]
            elif mode == "farthest":
                sel_local = np.argsort(d_end)[-k:]
            elif mode == "random":
                sel_local = rng.choice(len(idx), size=k, replace=False)
            else:
                raise ValueError("overlay_mode must be 'closest', 'farthest', or 'random'")
            chosen.extend(np.array(idx)[sel_local])
        return sorted(chosen)

    def _scatter_all_endpoints(self, ax, pcs_trials, y_num, s=16, a=0.35, n_components=None, name_by_label=None):
        """Scatter ENDPOINT (t=T-1) for all trials (2D or 3D)."""
        uniq = np.unique(y_num)
        for c in uniq:
            ends = pcs_trials[y_num == c, -1, :]  # [Nc, C]
            col = self.color_map_.get(c, (0.5, 0.5, 0.5, 0.6))  # neutral if unseen at fit
            if n_components == 2:
                ax.scatter(ends[:, 0], ends[:, 1], s=s, marker='o', alpha=a, color=col) #, label=f"class {name_by_label[c]}")
            else:
                ax.scatter(ends[:, 0], ends[:, 1], ends[:, 2], s=s, marker='o', alpha=a, color=col) #, label=f"class {name_by_label[c]}")

    def _plot_exemplars(self, ax, pcs_trials, y_num, idxs, alpha=0.35, extra_point_timestep=None, n_components=None, name_by_label=None):
        """Plot exemplar trajectories for provided indices (2D or 3D)."""
        for i in idxs:
            c = y_num[i]
            col = self.color_map_.get(c, (0.5, 0.5, 0.5, 0.6))
            if n_components == 2:
                xy = pcs_trials[i, :, :2]
                ax.plot(xy[:, 0], xy[:, 1], '-', lw=1.0, alpha=alpha, color=col) #, label=f"class {name_by_label[c]}")
                ax.scatter(xy[0, 0], xy[0, 1], s=80, marker='*', color=col, alpha=alpha)
                ax.scatter(xy[-1, 0], xy[-1, 1], s=80, marker='o', color=col, alpha=alpha)
                if extra_point_timestep is not None:
                    ax.scatter(xy[extra_point_timestep, 0], xy[extra_point_timestep, 1], s=80, marker='o', color=col, alpha=alpha)
            else:
                xyz = pcs_trials[i, :, :3]
                ax.plot(xyz[:, 0], xyz[:, 1], xyz[:, 2], '-', lw=1.0, alpha=alpha, color=col) #, label=f"class {name_by_label[c]}")
                ax.scatter(xyz[0, 0], xyz[0, 1], xyz[0, 2], s=80, marker='*', color=col, alpha=alpha)
                ax.scatter(xyz[-1, 0], xyz[-1, 1], xyz[-1, 2], s=80, marker='o', color=col, alpha=alpha)
                if extra_point_timestep is not None:
                    ax.scatter(xyz[extra_point_timestep, 0], xyz[extra_point_timestep, 1], xyz[extra_point_timestep, 2], s=80, marker='o', color=col, alpha=alpha)

    # ----------------------- Plot -----------------------

    def plot_variance_explained(self, activations=None, decisions=None, show=True):
        """Plot the variance explained by each principal component."""
        if not self.fitted_:
            raise RuntimeError("Call .fit(...) before .plot_variance_explained(...)")
        pca = self.pca
        explained_variance = pca.explained_variance_ratio_
        plt.plot(explained_variance)
        plt.xlabel("Principal Component")
        plt.ylabel("Variance Explained")
        if show:
            plt.show()
        return explained_variance

    def plot(
        self,
        activations,
        decisions,
        *,
        n_components=None,
        title=None,
        balance=False,                # balance for the *plotting subset* (does not change PCA)
        per_class_mean=True,
        plot_trajectory=True,
        collapse_to_class_means=False,
        overlay_examples_k=0,
        overlay_mode="closest",       # 'closest'|'farthest'|'random'
        overlay_classes=None,         # list of class values from decisions; None => all fitted classes
        plot_all_endpoints=False,
        alpha=0.5,
        lw=1.5,
        random_state=None,            # optional override for picking exemplars
        show=True,                    # set False if you want to compose multiple plots
        return_data=False,            # return (pcs_used, pcs_means or None),
        extra_point_timestep=None,
        ax=None,                       # NEW: axes to plot into, if not None
        label_segments_with_timesteps=False,  # NEW: If True, label each segment with its timestep (start-end)
    ):
        """
        Plot trajectories for a (possibly different) dataset using the already-fitted PCA.
        n_components: number of PCs to use for plotting (2, 3, or more, up to max_components fit at .fit time).
        ax: matplotlib Axes (2D or 3D) to plot into. If None, a new figure/axes is created.
        label_segments_with_timesteps: If True, label each segment (between two points) with its timestep index.
        """
        assert self.fitted_, "Call .fit(...) before .plot(...)."

        # Default n_components: 2 if not specified
        if n_components is None:
            n_components = 2
        if n_components > self.pca.components_.shape[0]:
            raise ValueError(f"Requested n_components={n_components} > max_components fit ({self.pca.components_.shape[0]})")

        rng = np.random.default_rng(self.random_state if random_state is None else random_state)
        A, y, N, T, D = self._check_inputs(activations, decisions)

        # Map y (values) to fit-time class indices; unseen classes get index -1
        y_idx = np.array([self.class_to_index_.get(v, -1) for v in y], dtype=int)
        if np.any(y_idx < 0):
            # Warn once in a non-intrusive way (comment-friendly)
            # print("[PCTrajectoryPlotter] Warning: some classes were unseen at fit; using neutral color.")
            pass

        # Optional balancing for plotting set only
        if balance and not collapse_to_class_means:
            keep = self._balance_indices(np.clip(y_idx, 0, None), rng)  # clip to avoid negatives in bincount
            A = A[keep]
            y_idx = y_idx[keep]
            N = A.shape[0]

        # Transform with fitted PCA
        pcs_trials = self.transform(A, n_components=n_components)  # (N, T, n_components)

        # Legend labels based on fit-time classes only
        uniq_fit = np.arange(len(self.classes_))
        name_by_label = {i: str(self.classes_[i]) for i in uniq_fit}

        # Resolve overlay class set (in fit-time index space)
        overlay_class_indices = self._resolve_overlay_class_indices(overlay_classes)

        # -- Collapsed to class means --
        if collapse_to_class_means:
            # Build means over *current* data by fit-time class index
            uniq_current = np.unique(y_idx[y_idx >= 0])  # only known classes
            if len(uniq_current) == 0:
                raise ValueError("No samples with classes seen at fit-time; cannot compute class means.")
            T_ = pcs_trials.shape[1]
            pcs_means = np.zeros((len(uniq_fit), T_, n_components), dtype=pcs_trials.dtype)

            for c in uniq_fit:
                idx = np.where(y_idx == c)[0]
                if len(idx) > 0:
                    pcs_means[c] = pcs_trials[idx].mean(axis=0)
                else:
                    pcs_means[c] = np.nan  # no samples for this class in current data

            if n_components == 2:
                if ax is None:
                    fig, ax_ = plt.subplots(figsize=(6, 4))
                    ax = ax_
                # Exemplars first (subset of classes), if requested
                if overlay_examples_k > 0:
                    idxs = self._pick_indices_per_class(np.clip(y_idx, 0, None), pcs_trials,
                                                        overlay_examples_k, overlay_mode,
                                                        overlay_class_indices, rng)
                    self._plot_exemplars(ax, pcs_trials, np.clip(y_idx, 0, None), idxs, alpha=min(alpha, 0.75), extra_point_timestep=extra_point_timestep, n_components=n_components, name_by_label=name_by_label)
                # All endpoints cloud
                if plot_all_endpoints:
                    self._scatter_all_endpoints(ax, pcs_trials, np.clip(y_idx, 0, None), s=18, a=0.75, n_components=n_components)

                for c in uniq_fit:
                    col = self.color_map_.get(c, (0.6, 0.6, 0.6, 0.9))
                    m = pcs_means[c]
                    if np.isnan(m).any():
                        continue
                    ax.plot(m[:, 0], m[:, 1], '-', lw=3, color=col, label=f"class {name_by_label[c]} (mean)")
                    ax.scatter(m[0, 0], m[0, 1], s=80, marker='*', color=col)
                    ax.scatter(m[-1, 0], m[-1, 1], s=80, marker='o', color=col)
                    # Label segments with timesteps if requested
                    # if label_segments_with_timesteps:
                    #     for t in range(m.shape[0] - 1):
                    #         x0, y0 = m[t, 0], m[t, 1]
                    #         x1, y1 = m[t + 1, 0], m[t + 1, 1]
                    #         xm, ym = (x0 + x1) / 2, (y0 + y1) / 2
                    #         ax.text(xm, ym, f"{t}-{t+1}", fontsize=8, color=col, alpha=0.7, ha='center', va='center')
                ax.set_xlabel("PC 1"); ax.set_ylabel("PC 2")
                ax.set_title(title or "Canonical class trajectories + exemplars")
                ax.legend(frameon=False)
                plt.tight_layout()
                if show and ax is not None and getattr(ax, 'figure', None) is not None:
                    plt.show()
            elif n_components == 3:
                from mpl_toolkits.mplot3d import Axes3D  # noqa
                if ax is None:
                    fig = plt.figure(figsize=(10, 8))
                    ax_ = fig.add_subplot(111, projection='3d')
                    ax = ax_
                if overlay_examples_k > 0:
                    idxs = self._pick_indices_per_class(np.clip(y_idx, 0, None), pcs_trials,
                                                        overlay_examples_k, overlay_mode,
                                                        overlay_class_indices, rng)
                    self._plot_exemplars(ax, pcs_trials, np.clip(y_idx, 0, None), idxs, alpha=min(alpha, 0.75), n_components=n_components, name_by_label=name_by_label)
                if plot_all_endpoints:
                    self._scatter_all_endpoints(ax, pcs_trials, np.clip(y_idx, 0, None), s=18, a=min(alpha, 0.75), n_components=n_components, name_by_label=name_by_label)

                for c in uniq_fit:
                    col = self.color_map_.get(c, (0.6, 0.6, 0.6, 0.9))
                    m = pcs_means[c]
                    if np.isnan(m).any():
                        continue
                    ax.plot(m[:, 0], m[:, 1], m[:, 2], '-', lw=3, color=col,
                            label=f"class {name_by_label[c]} (mean)")
                    ax.scatter(m[0, 0], m[0, 1], m[0, 2], s=80, marker='*', color=col)
                    ax.scatter(m[-1, 0], m[-1, 1], m[-1, 2], s=80, marker='o', color=col)
                    # Label segments with timesteps if requested
                    # if label_segments_with_timesteps:
                    #     for t in range(m.shape[0] - 1):
                    #         x0, y0, z0 = m[t, 0], m[t, 1], m[t, 2]
                    #         x1, y1, z1 = m[t + 1, 0], m[t + 1, 1], m[t + 1, 2]
                    #         xm, ym, zm = (x0 + x1) / 2, (y0 + y1) / 2, (z0 + z1) / 2
                    #         ax.text(xm, ym, zm, f"{t}-{t+1}", fontsize=8, color=col, alpha=0.7, ha='center', va='center')
                ax.set_xlabel("PC 1"); ax.set_ylabel("PC 2"); ax.set_zlabel("PC 3")
                ax.set_title(title or "Canonical class trajectories + exemplars")
                plt.tight_layout()
                if show and ax is not None and getattr(ax, 'figure', None) is not None:
                    plt.show()
            else:
                raise ValueError("Only 2D or 3D plotting is supported for collapsed class means.")
            return (pcs_trials, pcs_means) if return_data else None

        # -- Non-collapsed trajectories --
        # Choose exemplar indices (subset of classes)
        idxs = []
        if overlay_examples_k > 0:
            idxs = self._pick_indices_per_class(np.clip(y_idx, 0, None), pcs_trials,
                                                overlay_examples_k, overlay_mode,
                                                overlay_class_indices, rng)

        if n_components == 2:
            if ax is None:
                fig, ax_ = plt.subplots(figsize=(8, 6))
                ax = ax_
            # Plot the exemplar trajectories
            if idxs:
                self._plot_exemplars(ax, pcs_trials, np.clip(y_idx, 0, None), idxs, alpha=min(alpha, 0.75), n_components=n_components, name_by_label=name_by_label)

            # Plot the selected (same exemplar set) or all? — follow original: draw only chosen exemplars
            # But add class means if requested (computed over all trials in current data)
            if per_class_mean:
                for c in np.unique(np.clip(y_idx, 0, None)):
                    if c < 0:  # skip unseen
                        continue
                    m = pcs_trials[y_idx == c].mean(axis=0)  # (T, C)
                    col = self.color_map_.get(c, (0.6, 0.6, 0.6, 0.9))
                    ax.plot(m[:, 0], m[:, 1], '-', lw=3, color=col, label=f"class {name_by_label[c]} (mean)")
                    # # Label segments with timesteps if requested
                    if label_segments_with_timesteps and not hasattr(ax, "_timesteps_labeled"):
                        print("labeling timesteps")
                        for t in range(0, m.shape[0] - 1, 2):
                            x0, y0 = m[t, 0], m[t, 1]
                            x1, y1 = m[t + 1, 0], m[t + 1, 1]
                            xm, ym = (x0 + x1) / 2, (y0 + y1) / 2
                            ax.text(xm, ym, f"{t}-{t+1}", fontsize=8, color=col, alpha=0.7, ha='center', va='center')
                        ax._timesteps_labeled = True

            # Optional: cloud of all endpoints
            if plot_all_endpoints:
                self._scatter_all_endpoints(ax, pcs_trials, np.clip(y_idx, 0, None), s=16, a=min(alpha, 0.75), n_components=n_components)

            ax.set_xlabel("PC 1"); ax.set_ylabel("PC 2")
            
        elif n_components == 3:
            from mpl_toolkits.mplot3d import Axes3D  # noqa
            if ax is None:
                fig = plt.figure(figsize=(10, 8))
                ax_ = fig.add_subplot(111, projection='3d')
                ax = ax_
            if idxs:
                self._plot_exemplars(ax, pcs_trials, np.clip(y_idx, 0, None), idxs, alpha=min(alpha, 0.75), n_components=n_components, name_by_label=name_by_label)

            if per_class_mean:
                for c in np.unique(np.clip(y_idx, 0, None)):
                    if c < 0:  # skip unseen
                        continue
                    m = pcs_trials[y_idx == c].mean(axis=0)  # (T, C)
                    col = self.color_map_.get(c, (0.6, 0.6, 0.6, 0.9))
                    ax.plot(m[:, 0], m[:, 1], m[:, 2], '-', lw=3, color=col,
                            label=f"class {name_by_label[c]} (mean)")
                    # Label segments with timesteps if requested
                    # if label_segments_with_timesteps:
                    #     for t in range(m.shape[0] - 1):
                    #         x0, y0, z0 = m[t, 0], m[t, 1], m[t, 2]
                    #         x1, y1, z1 = m[t + 1, 0], m[t + 1, 1], m[t + 1, 2]
                    #         xm, ym, zm = (x0 + x1) / 2, (y0 + y1) / 2, (z0 + z1) / 2
                    #         ax.text(xm, ym, zm, f"{t}-{t+1}", fontsize=8, color=col, alpha=0.7, ha='center', va='center')
                
            if plot_all_endpoints:
                self._scatter_all_endpoints(ax, pcs_trials, np.clip(y_idx, 0, None), s=16, a=min(alpha, 0.75), n_components=n_components)
            
            ax.set_xlabel("PC 1"); ax.set_ylabel("PC 2"); ax.set_zlabel("PC 3")
            
        # Only plot one legend entry per class, using name_by_label
        handles = []
        labels = []
        for c in np.unique(np.clip(y_idx, 0, None)):
            if c < 0:
                continue
            col = self.color_map_.get(c, (0.6, 0.6, 0.6, 0.9))
            # Create a proxy artist for the legend
            # For 2D and 3D, handle legend proxy accordingly
            if n_components == 2:
                h = ax.plot([], [], '-', lw=3, color=col)[0]
            elif n_components == 3:
                h = ax.plot([], [], [], '-', lw=3, color=col)[0]
            else:
                continue
            handles.append(h)
            labels.append(f"class {name_by_label[c]}")
        if handles:
            ax.legend(handles, labels, frameon=False)
        
        ax.set_title(title or "Trajectories in PC space")
        plt.tight_layout()
        if show and ax is not None and getattr(ax, 'figure', None) is not None:
            plt.show()
        elif not (n_components == 2 or n_components == 3):
            raise ValueError("Only 2D or 3D plotting is supported for non-collapsed trajectories.")

        return pcs_trials if return_data else None

# --- UMAP-based variant that reuses the PCA plotter's API and plotting logic ---
class UMAPTrajectoryPlotter(PCTrajectoryPlotter):
    """
    Same interface as PCTrajectoryPlotter, but uses UMAP (umap-learn) as the reducer.
    You can fit once and then plot multiple different datasets with the same embedding.

    Notes:
      - Requires `umap-learn>=0.5` for `.transform()` on new data.
      - Arguments mirror umap.UMAP common params; pass extras via umap_kwargs.
    """

    def __init__(
        self,
        max_components=None,
        n_components=None,
        random_state=0,
        n_neighbors=15,
        min_dist=0.1,
        metric="euclidean",
        umap_kwargs=None,
    ):
        # initialize base (for shared fields, colors, metadata, plotting helpers)
        # For backward compatibility, n_components is an alias for max_components
        if max_components is None and n_components is not None:
            max_components = n_components
        super().__init__(max_components=max_components, random_state=random_state)

        try:
            import umap
        except ImportError as e:
            raise ImportError(
                "UMAPTrajectoryPlotter requires the `umap-learn` package.\n"
                "Install with: pip install umap-learn"
            ) from e

        # Build the UMAP reducer
        self._umap_mod = umap
        self.reducer = umap.UMAP(
            n_components=max_components if max_components is not None else 2,
            n_neighbors=n_neighbors,
            min_dist=min_dist,
            metric=metric,
            random_state=random_state,
            **(umap_kwargs or {}),
        )
        self.reducer_name_ = "umap"
        # Keep a flag like the PCA class had (how we fit: collapsed or not)
        self.fit_collapsed_ = False
        self.max_components = max_components if max_components is not None else 2

    # -------- override fit/transform to use UMAP instead of PCA --------

    def fit(self, activations, decisions, *, balance_for_fit=False, collapse_to_class_means=False):
        """
        Fit UMAP on (N,T,D) activations.
          - balance_for_fit=True balances class counts before building the UMAP graph.
          - collapse_to_class_means=True fits UMAP on per-class mean trajectories only.
        """
        rng = np.random.default_rng(self.random_state)
        A, y, N, T, D = self._check_inputs(activations, decisions)
        self._fit_D = D

        # establish classes at fit time (same as PCA version)
        classes, y_num = np.unique(y, return_inverse=True)
        self.classes_ = classes
        self.class_to_index_ = {c: i for i, c in enumerate(classes)}

        if collapse_to_class_means:
            A_means = []
            for c in np.unique(y_num):
                idx = np.where(y_num == c)[0]
                A_means.append(A[idx].mean(axis=0))  # (T,D)
            A_fit = np.stack(A_means, axis=0)  # (C,T,D)
            X_fit = A_fit.reshape(-1, D)
            self.fit_collapsed_ = True
        else:
            if balance_for_fit:
                counts = np.bincount(y_num)
                k_bal = counts[counts > 0].min()
                keep = np.concatenate([
                    rng.choice(np.where(y_num == c)[0], k_bal, replace=False)
                    for c in np.unique(y_num)
                ])
                keep.sort()
                A = A[keep]
            X_fit = A.reshape(-1, D)
            self.fit_collapsed_ = False

        # Fit UMAP on flattened (time) data
        self.reducer.fit(X_fit)

        # lock class colors as in PCA version
        uniq_indices = np.arange(len(classes))
        self.color_map_ = self._build_color_map(uniq_indices)

        self.fitted_ = True
        return self

    def transform(self, activations, n_components=None):
        """
        Transform (N,T,D) activations into the fitted UMAP space.
        Returns (N,T,n_components).
        n_components: number of UMAP components to return (<= max_components fit at .fit time).
        """
        assert self.fitted_, "Call .fit(...) first."
        A = np.asarray(activations)
        assert A.ndim == 3
        N, T, D = A.shape
        if D != self._fit_D:
            raise ValueError(f"Input feature dimension {D} does not match fit-time dimension {self._fit_D}")

        # UMAP transform (requires umap-learn >= 0.5)
        try:
            X = A.reshape(-1, D)
            emb = self.reducer.transform(X)  # (N*T, max_components)
        except AttributeError as e:
            raise RuntimeError(
                "Your umap-learn version does not support `.transform()`.\n"
                "Upgrade with: pip install --upgrade umap-learn"
            ) from e

        # Determine how many components to return
        max_components_fit = emb.shape[1]
        if n_components is None:
            n_components = max_components_fit
        if n_components > max_components_fit:
            raise ValueError(f"Requested n_components={n_components} > max_components fit ({max_components_fit})")
        return emb.reshape(N, T, max_components_fit)[..., :n_components]


In [ ]:
def keep_only_label_from_loader(loader, target_labels, batch_size=None, shuffle=False):
    xs, ys = [], []
    for x, y in loader:
        # make labels 1-D over batch
        if y.ndim > 1:
            # if one-hot/probabilities, reduce to class id
            if y.size(-1) > 1:
                y_flat = y.argmax(dim=-1)
            else:
                y_flat = y.squeeze(-1)
        else:
            y_flat = y

        mask = False
        for label in target_labels:
            mask = mask | (y_flat == label)
        if mask.any():
            xs.append(x[mask, ...])                # index batch dim only
            ys.append(y_flat[mask])

    if not xs:
        raise ValueError(f"No samples with label {target_label}.")

    X = torch.cat(xs, dim=0)
    Y = torch.cat(ys, dim=0)
    ds = TensorDataset(X, Y)
    return DataLoader(
        ds,
        batch_size or loader.batch_size,
        shuffle=shuffle,
        num_workers=loader.num_workers,
        pin_memory=loader.pin_memory,
        drop_last=loader.drop_last,
    )

def _balance_classes(activations, decisions, seed=42):
    """
    Downsample each class to match the smallest class size.
    """
    rng = np.random.default_rng(seed)
    activations = np.asarray(activations)
    decisions = np.asarray(decisions)

    if decisions.shape[0] != activations.shape[0]:
        raise ValueError("decisions must have length == activations.shape[0] (one label per row).")

    unique_classes, counts = np.unique(decisions, return_counts=True)
    min_count = counts.min()

    balanced_idx = []
    for cls in unique_classes:
        cls_idx = np.where(decisions == cls)[0]
        chosen = rng.choice(cls_idx, min_count, replace=False)
        balanced_idx.append(chosen)

    balanced_idx = np.concatenate(balanced_idx)
    rng.shuffle(balanced_idx)

    return activations[balanced_idx], decisions[balanced_idx]
    
def decision_decoder(activations, decisions, test=None, balance=True):
    """
    Decodes the decision from the activations using a logistic regression model.
    Optionally balances class counts before training.
    """
    if balance:
        activations, decisions = _balance_classes(activations, decisions)

    model = LogisticRegression(max_iter=1000)
    model.fit(activations, decisions)

    if test is None:
        accuracy = model.score(activations, decisions)
    else:
        accuracy = model.score(test[0], test[1])

    return model, accuracy

class MLPDecoder(nn.Module):
    def __init__(self, D, hidden=128, out_dim=8, dropout=0.0):
        super().__init__()
        self.fc1 = nn.Linear(D, hidden)
        self.drop = nn.Dropout(dropout) if dropout > 0 else nn.Identity()
        self.fc2 = nn.Linear(hidden, out_dim)
    def forward(self, x):                # x: [N, D]
        h = F.relu(self.fc1(x))
        h = self.drop(h)
        return self.fc2(h)               # logits or scalar(s)
    def project_hidden(self, x):         # get hidden features [N, hidden]
        return F.relu(self.fc1(x))


def train_mlp_decoder(
    X, y, X_test, y_t_test, *, hidden=128, task="multiclass", out_dim=None,
    epochs=20, bs=128, lr=1e-3, weight_decay=0.0, device=None, dropout=0.0
):
    """
    Trains an MLP (input -> hidden -> output) and returns the model with the LOWEST val loss.
    X: float [N,D], y: [N]
    X_test, y_t_test: validation set
    """
    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"
    N, D = X.shape
    if out_dim is None:
        if task == "multiclass":
            C_train = int(y.max().item()) + 1
            C_val   = int(y_t_test.max().item()) + 1
            out_dim = max(C_train, C_val)
        else:
            out_dim = 1

    model = MLPDecoder(D, hidden=hidden, out_dim=out_dim, dropout=dropout).to(device)

    # loss + label shaping
    def prep_y(lbl):
        if task == "multiclass":
            return lbl.long()
        elif task == "binary":
            return lbl.float().unsqueeze(1)
        elif task == "regression":
            return lbl.float().unsqueeze(1) if out_dim == 1 else lbl.float()
        else:
            raise ValueError("task must be 'multiclass'|'binary'|'regression'")

    y_tr  = prep_y(y)
    y_val = prep_y(y_t_test)

    if task == "multiclass":
        crit = nn.CrossEntropyLoss()
    elif task == "binary":
        crit = nn.BCEWithLogitsLoss()
    else:
        crit = nn.MSELoss()

    tr_dl = DataLoader(TensorDataset(X, y_tr), batch_size=bs, shuffle=True)
    va_dl = DataLoader(TensorDataset(X_test, y_val), batch_size=bs, shuffle=False)

    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

    best_val = float("inf")
    best_state = None
    best_epoch = -1

    for ep in tqdm(range(1, epochs + 1)):
        # ---- train
        model.train()
        for xb, yb in tr_dl:
            xb, yb = xb.to(device), yb.to(device)
            logits = model(xb)
            loss = crit(logits, yb)
            opt.zero_grad(set_to_none=True)
            loss.backward()
            opt.step()

        # ---- validate
        model.eval()
        val_sum, n_batches = 0.0, 0
        with torch.no_grad():
            for xb, yb in va_dl:
                xb, yb = xb.to(device), yb.to(device)
                logits = model(xb)
                val_sum += crit(logits, yb).item()
                n_batches += 1
        val_loss = val_sum / max(1, n_batches)

        # track best
        if val_loss < best_val - 1e-9:
            best_val = val_loss
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            best_epoch = ep

        # optional: print progress
        # print(f"epoch {ep}: val_loss={val_loss:.4f} (best={best_val:.4f} @ {best_epoch})")

    # restore best weights
    if best_state is not None:
        model.load_state_dict(best_state)

    return model, {"best_val_loss": best_val, "best_epoch": best_epoch}
    
import torch, torch.nn as nn

@torch.no_grad()
def hidden_from_mlp(model, X, *, apply_relu=True, batch_size=0, device=None):
    """
    X: [N, D]  ->  returns H: [N, hidden]
    Looks for model.fc1 (nn.Linear); falls back to the first Linear found.
    """
    model.eval()
    if device is None:
        device = next(model.parameters()).device

    # pick the linear layer that defines the hidden space
    lin = getattr(model, "fc1", None)
    if not isinstance(lin, nn.Linear):
        lin = next(m for m in model.modules() if isinstance(m, nn.Linear))

    def run_chunk(xb):
        h = lin(xb.to(device))
        return (torch.relu(h) if apply_relu else h).cpu()

    if not batch_size:
        return run_chunk(X)

    outs = []
    for s in range(0, X.size(0), batch_size):
        outs.append(run_chunk(X[s:s+batch_size]))
    return torch.cat(outs, dim=0)

# USE MODEL's READOUT WEIGHTS TO ON ACTIVATIONS OVER AT ANY STEP

def batched_readout_from_extracted(
    x,                 # (N, T, 8, H, W)
    conv_w, conv_b,    # [8, 8, 5, 5], [8]
    lin_w, lin_b,      # [512, 8], [512]
    use_relu=True,
    pool="max",        # "max" or "avg"
    padding=None       # None -> same-ish (k//2)
):
    """
    Returns (N, T, 512) from extracted weights:
      x -> conv2d(8->8,k=5) -> ReLU? -> Adaptive{pool}(1,1) -> Flatten -> Linear(8->512)
    """
    assert x.ndim == 5 and x.shape[2] == conv_w.shape[1] == 8, "Expect x:(N,T,8,H,W)"
    N, T, C, H, W = x.shape
    dev, dt = conv_w.device, conv_w.dtype

    # Flatten batch+time, move/dtype-match to weights
    x2 = x.reshape(N*T, C, H, W).to(device=dev, dtype=dt)

    # default "same" padding for odd kernel
    if padding is None:
        kH, kW = conv_w.shape[-2:]
        padding = (kH // 2, kW // 2)

    y = F.conv2d(x2, conv_w, conv_b, stride=1, padding=padding)
    if use_relu:
        y = F.relu(y)

    if pool == "max":
        y = F.adaptive_max_pool2d(y, (1, 1))
    elif pool == "avg":
        y = F.adaptive_avg_pool2d(y, (1, 1))
    else:
        raise ValueError("pool must be 'max' or 'avg'")

    y = y.view(N*T, -1)                 # (N*T, 8)
    out = F.linear(y, lin_w, lin_b)     # (N*T, 512)
    return out.view(N, T, -1)           # (N, T, 512)

In [ ]:
def map_distances(maze, start_goal, radius=2):
    H, W = maze.shape
    dist = np.full((H, W), np.inf, dtype=float)
    maze2 = maze.copy()
    q = deque()

    # seed from input blocks
    for (i,j), v in np.ndenumerate(start_goal):
        if v:
            dist[i,j] = 0
            maze2[i,j] = 1
            q.append((i,j))

    # all offsets in a (2*radius+1)^2 patch, excluding (0,0)
    nbrs = [(dx,dy)
            for dx in range(-radius, radius+1)
            for dy in range(-radius, radius+1)
            if not (dx==0 and dy==0)]

    while q:
        i, j = q.popleft()
        dnext = dist[i, j] + 1
        for dx, dy in nbrs:
            ii, jj = i + dx, j + dy
            if not (0 <= ii < H and 0 <= jj < W): 
                continue
            if maze2[ii, jj] != 1:
                continue

            # only block if both orthogonal neighbors are walls
            if dx != 0 and dy != 0:
                if maze2[i+dx, j] == 0 and maze2[i, j+dy] == 0:
                    continue

            if dist[ii, jj] == np.inf:
                dist[ii, jj] = dnext
                q.append((ii, jj))

    return dist

def bfs_distance(maze, seeds):
    H, W = maze.shape
    dist = np.full((H, W), np.inf, dtype=float)
    q = deque()

    # allow BFS to traverse the seeds even if maze[...] == 0
    maze2 = maze.copy()
    for (i, j) in seeds:
        dist[i, j] = 0
        maze2[i, j] = 1
        q.append((i, j))

    nbrs = [(1,0),(-1,0),(0,1),(0,-1)]
    while q:
        i, j = q.popleft()
        for di, dj in nbrs:
            ii, jj = i+di, j+dj
            if 0 <= ii < H and 0 <= jj < W:
                if maze2[ii, jj] == 1 and dist[ii, jj] == np.inf:
                    dist[ii, jj] = dist[i, j] + 1
                    q.append((ii, jj))
    return dist

#THIS ONE GIVES ACTUAL SHORTEST PATH
def compute_shortest_path_mask_actual(maze, start_goal):
    """
    Returns an (H,W) uint8 mask marking corridor cells on any shortest path
    between the two 2×2 input blocks. If the blocks are disconnected, returns
    all zeros.
    """
    H, W = maze.shape

    blocks = get_start_coordinates(maze, start_goal)
    
    if len(blocks) != 2:
        return np.zeros((H, W), dtype=np.uint8)

    # 2) BFS distances from each block
    dist0 = bfs_distance(maze, blocks[0])
    dist1 = bfs_distance(maze, blocks[1])

    # 3) Minimal distance between blocks
    d_min = min(dist0[i,j] for (i,j) in blocks[1])

    # 4) If infinite, no path exists → return zero mask
    if np.isinf(d_min):
        return np.zeros((H, W), dtype=np.uint8)

    # 5) Mark cells where dist0 + dist1 == d_min and are corridors
    sp = ((dist0 + dist1) == d_min) & (maze == 1)
    return sp.astype(np.uint8)
    

#THIS ONE GIVES SHORTEST PATH AT CORRIDOR RESOLUTION
# def compute_shortest_path_mask(maze, start_goal):
#     """
#     Corridors are 2 px wide. Do shortest-path at corridor resolution by:
#       - downsample: take every other pixel
#       - shortest-path on coarse grid
#       - upsample: repeat back to 2x2 blocks
#     """
#     H, W = maze.shape
#     maze01 = (maze == 1).astype(np.uint8)
#     sg01   = (start_goal != 0).astype(np.uint8)

#     # 1) Downsample by taking every other pixel
#     maze_small = maze01[::2, ::2].astype(np.uint8)

#     # 2) Downsample start/goal by OR-ing the 4 sub-samples (still very simple)
#     #    This collapses each original 2x2 start block to a single 1 on the coarse grid.
#     sg_small = sg01[::2, ::2]
#     sg_small = sg_small.astype(np.uint8)

#     # 3) Shortest path on coarse grid using your existing function
#     sp_small = compute_shortest_path_mask_actual(maze_small, sg_small).astype(np.uint8)

#     # 4) Upsample back to original resolution and clip to corridor pixels
#     sp_big = np.repeat(np.repeat(sp_small, 2, axis=0), 2, axis=1)
#     sp_big = sp_big[:H, :W].astype(np.uint8)         # crop if odd dims
#     shifted = np.zeros_like(sp_big)
#     shifted[:-1, :-1] = sp_big[1:, 1:]
#     final = np.clip(shifted - start_goal, 0, 1)

#     return final

import numpy as np
from collections import deque

def compute_shortest_path_mask(maze: np.ndarray, start_goal: np.ndarray) -> np.ndarray:
    """
    maze: 2D array (0 = wall, 1 = corridor)
    start_goal: 2D array same shape as maze, with exactly two 1s for the endpoints
    returns: 2D array of 0/1 with 1s along the shortest path (inclusive of endpoints).
             If no path, returns all zeros.
    """
    maze = (maze != 0).astype(np.uint8)
    sg = (start_goal != 0).astype(np.uint8)

    # Find the two start/goal positions
    pts = np.argwhere(sg == 1)
    if pts.shape[0] != 2:
        # raise ValueError(f"start_goal must contain exactly two 1s; got {pts.shape[0]}")
        return start_goal
    s = tuple(pts[0])
    t = tuple(pts[1])

    # If start/goal are on walls, treat them as corridors for the search
    maze[s] = 1
    maze[t] = 1

    H, W = maze.shape
    visited = np.zeros_like(maze, dtype=bool)
    parent = -np.ones((H, W, 2), dtype=np.int32)  # store parent coords

    # 4-neighborhood (up, down, left, right)
    nbrs = [(-1,0),(1,0),(0,-1),(0,1)]

    q = deque([s])
    visited[s] = True

    found = False
    while q:
        r, c = q.popleft()
        if (r, c) == t:
            found = True
            break
        for dr, dc in nbrs:
            nr, nc = r + dr, c + dc
            if 0 <= nr < H and 0 <= nc < W and not visited[nr, nc] and maze[nr, nc] == 1:
                visited[nr, nc] = True
                parent[nr, nc] = (r, c)
                q.append((nr, nc))

    path_map = np.zeros_like(maze, dtype=np.uint8)
    if not found:
        return path_map  # no path

    # Reconstruct from t back to s
    cur = t
    while True:
        path_map[cur] = 1
        if cur == s:
            break
        pr, pc = parent[cur]
        if pr == -1:  # safety (shouldn't happen if found==True)
            path_map[:] = 0
            break
        cur = (int(pr), int(pc))

    return path_map.astype(np.uint8)

import numpy as np
from collections import deque
from typing import Literal, Tuple

def compute_reachable_mask(
    maze: np.ndarray,
    start_goal: np.ndarray,
    connectivity: Literal[4, 8] = 4,
    return_individual: bool = False,
):
    """
    Return a 0/1 map of all corridor cells reachable from ANY start in `start_goal`.

    maze: 2D array (0 = wall, 1 = corridor)
    start_goal: 2D array same shape; 1s indicate start points (2 typical, but N>=1 allowed)
    connectivity: 4 or 8 neighbor connectivity
    return_individual: if True and there are exactly 2 starts, also return separate maps
                       for each start (tuple: union_map, map_from_s1, map_from_s2)

    Returns:
      reach_map if return_individual=False
      (reach_map, map1, map2) if return_individual=True and exactly 2 starts
    """
    H, W = maze.shape
    corridors = (maze != 0)  # bool
    seeds = np.argwhere(start_goal != 0)
    # if seeds.size != 2:
    #     print("touching")
    #     plt.imshow(start_goal)
    #     plt.show()
    #     return np.zeros_like(maze)

    # If any start is on a wall, allow stepping from it by treating as corridor
    corridors = corridors.copy()
    for r, c in seeds:
        corridors[r, c] = True

    if connectivity == 4:
        nbrs = [(-1,0), (1,0), (0,-1), (0,1)]
    elif connectivity == 8:
        nbrs = [(-1,0), (1,0), (0,-1), (0,1), (-1,-1), (-1,1), (1,-1), (1,1)]
    else:
        raise ValueError("connectivity must be 4 or 8")

    # Multi-source BFS for union reachability
    visited = np.zeros((H, W), dtype=bool)
    q = deque()
    for r, c in seeds:
        if 0 <= r < H and 0 <= c < W:
            visited[r, c] = True
            q.append((r, c))
    while q:
        r, c = q.popleft()
        for dr, dc in nbrs:
            nr, nc = r + dr, c + dc
            if 0 <= nr < H and 0 <= nc < W and not visited[nr, nc] and corridors[nr, nc]:
                visited[nr, nc] = True
                q.append((nr, nc))

    reach_map = visited.astype(np.uint8)

    if return_individual:
        return reach_map.astype(np.uint8)

    # If asked, compute separate reachability from each of the two starts
    def bfs_from(seed: Tuple[int, int]) -> np.ndarray:
        vis = np.zeros((H, W), dtype=bool)
        dq = deque([tuple(seed)])
        vis[tuple(seed)] = True
        while dq:
            r, c = dq.popleft()
            for dr, dc in nbrs:
                nr, nc = r + dr, c + dc
                if 0 <= nr < H and 0 <= nc < W and not vis[nr, nc] and corridors[nr, nc]:
                    vis[nr, nc] = True
                    dq.append((nr, nc))
        return vis.astype(np.uint8)

    map1 = bfs_from(tuple(seeds[0]))
    map2 = bfs_from(tuple(seeds[1]))
    return reach_map, map1, map2


# def compute_reachable_mask(maze, start_goal, t):
#     """
#     All cells reachable within t conv‐hops.
#     """
#     dist_map = map_distances(maze, start_goal)
#     return (dist_map <= t).astype(np.uint8)

def compute_node_degree(maze, start_goal):
    """
    For each corridor cell, count 4‐connected corridor neighbors.
    """
    H, W = maze.shape
    deg = np.zeros((H,W), int)
    for (i,j), v in np.ndenumerate(maze):
        if v==1:
            cnt = 0
            for di,dj in [(1,0),(-1,0),(0,1),(0,-1)]:
                ii, jj = i+di, j+dj
                if 0<=ii<H and 0<=jj<W and maze[ii,jj]==1:
                    cnt += 1
            deg[i,j] = cnt
    return deg

    
def compute_degree(maze, path):
    H, W = maze.shape
    deg = np.zeros((H,W), int)
    for (i,j), v in np.ndenumerate(maze):
        if v==1:
            cnt = 0
            for di,dj in [(1,0),(-1,0),(0,1),(0,-1)]:
                ii, jj = i+di, j+dj
                if 0<=ii<H and 0<=jj<W and path[ii,jj]==1:
                    cnt += 1
            deg[i,j] = cnt
    return np.sum(deg)


def collect_degrees(dataloader):
    degrees_in_reachable = []
    degrees_in_shortest = []
    shortest_path_lengths = []
    reachable_path_lengths = []
    degrees_in_reachable_from_shortest = []
    reachable_from_shortest_lengths = []
    sample_idx = 0

    with torch.no_grad():
        for batch_inputs, _labels in tqdm(dataloader):
            for input in batch_inputs:
                input = np.array(input)
                maze = input[0]
                start_goal = input[-1]

                maze01 = (maze == 1)
                sg01   = (start_goal != 0)
                maze = maze01[::2, ::2]
                start_goal = sg01[::2, ::2]

                reachable = compute_reachable_mask(maze, start_goal, 400)
                deg = compute_degree(maze, start_goal, reachable)
                degrees_in_reachable.append(deg)
                reachable_path_lengths.append(np.sum(reachable))

                shortest_path = compute_shortest_path_mask(maze, start_goal)
                deg = compute_degree(maze, start_goal, shortest_path)
                degrees_in_shortest.append(deg)
                shortest_path_lengths.append(np.sum(shortest_path))

                total_len, reach_mask, sp_mask = between_reachable_from_shortest(maze, start_goal, include_shortest_in_count=False)
                deg = compute_degree(maze, start_goal, reach_mask)
                degrees_in_reachable_from_shortest.append(deg)
                reachable_from_shortest_lengths.append(total_len)

                sample_idx += 1

    return np.array(degrees_in_reachable), np.array(degrees_in_shortest), np.array(reachable_path_lengths), np.array(shortest_path_lengths), np.array(reachable_from_shortest_lengths), np.array(degrees_in_reachable_from_shortest)

import numpy as np
from collections import deque
from typing import Literal, Tuple, Dict

def compute_degree(
    maze: np.ndarray,
    path_map: np.ndarray,
    *,
    connectivity: Literal[4, 8] = 4,
    mode: Literal["regions", "mouths"] = "mouths",
    return_details: bool = False,
):
    """
    Count branches off a given path.

    maze: 2D int/bool (0=wall, 1=corridor)
    path_map: 2D int/bool (1s exactly on the path)
    connectivity: 4 or 8
    mode:
      - "regions": count each connected off-path corridor region that touches the path
      - "mouths":  count distinct attachment mouths (separates multiple touches from the same region)
    return_details: also return label map and sizes for debugging

    Returns:
      count
      or (count, labels, sizes) if return_details=True
        labels: int32 map, 0 = not counted, k>=1 = branch id
        sizes:  dict {branch_id: number_of_cells}
    """
    H, W = maze.shape
    corridors = (maze != 0)
    on_path   = (path_map != 0)

    # corridors we are allowed to traverse that are not the path itself
    offpath = corridors & (~on_path)

    if connectivity == 4:
        nbrs = [(-1,0), (1,0), (0,-1), (0,1)]
    elif connectivity == 8:
        nbrs = [(-1,0), (1,0), (0,-1), (0,1), (-1,-1), (-1,1), (1,-1), (1,1)]
    else:
        raise ValueError("connectivity must be 4 or 8")

    # Identify off-path cells adjacent to the path (the “boundary” or “mouth” cells)
    boundary = np.zeros_like(offpath, dtype=bool)
    pr, pc = np.where(on_path)
    for r, c in zip(pr, pc):
        for dr, dc in nbrs:
            nr, nc = r + dr, c + dc
            if 0 <= nr < H and 0 <= nc < W and offpath[nr, nc]:
                boundary[nr, nc] = True

    visited = np.zeros_like(offpath, dtype=bool)
    labels  = np.zeros_like(offpath, dtype=np.int32)
    sizes: Dict[int, int] = {}
    branch_id = 0

    # Seeds are boundary cells (adjacent to path)
    seeds = np.argwhere(boundary)

    for sr, sc in seeds:
        if visited[sr, sc]:
            continue

        branch_id += 1
        q = deque([(sr, sc)])
        visited[sr, sc] = True
        labels[sr, sc]  = branch_id
        size = 1

        while q:
            r, c = q.popleft()
            for dr, dc in nbrs:
                nr, nc = r + dr, c + dc
                if not (0 <= nr < H and 0 <= nc < W):
                    continue
                if visited[nr, nc] or not offpath[nr, nc]:
                    continue

                if mode == "regions":
                    # Flood whole off-path component; boundary cells are fine
                    pass
                elif mode == "mouths":
                    # Do NOT cross through other boundary cells (other mouths)
                    # except the seed itself. This isolates each attachment mouth.
                    if boundary[nr, nc] and not (nr == sr and nc == sc):
                        continue
                else:
                    raise ValueError("mode must be 'regions' or 'mouths'")

                visited[nr, nc] = True
                labels[nr, nc]  = branch_id
                size += 1
                q.append((nr, nc))

        sizes[branch_id] = size

        # In "regions" mode, starting from one boundary cell floods the entire
        # connected off-path region. All other boundary cells in that same region
        # will already be visited, so they won’t create extra branches. In "mouths"
        # mode we block crossing via other boundary cells, so distinct mouths are split.

    if return_details:
        return branch_id, labels, sizes
    return branch_id


def collect_degrees(dataloader):
    degrees_in_reachable = []
    degrees_in_shortest = []
    shortest_path_lengths = []
    reachable_path_lengths = []
    degrees_in_reachable_from_shortest = []
    reachable_from_shortest_lengths = []
    sample_idx = 0

    with torch.no_grad():
        for batch_inputs, _labels in tqdm(dataloader):
            for input in batch_inputs:
                input = np.array(input)
                maze = input[0]
                start_goal = input[-1]

                maze01 = (maze == 1)
                sg01   = (start_goal != 0)
                maze = maze01[::2, ::2]
                start_goal = sg01[::2, ::2]

                reachable = compute_reachable_mask(maze, start_goal, return_individual=True)
                deg = compute_degree(maze, reachable)
                degrees_in_reachable.append(deg)
                reachable_path_lengths.append(np.sum(reachable))

                shortest_path = compute_shortest_path_mask(maze, start_goal)
                deg = compute_degree(maze, shortest_path)
                degrees_in_shortest.append(deg)
                shortest_path_lengths.append(np.sum(shortest_path))

                # total_len, reach_mask, sp_mask = between_reachable_from_shortest(maze, start_goal, include_shortest_in_count=False)
                # deg = compute_degree(maze, start_goal, reach_mask)
                # degrees_in_reachable_from_shortest.append(deg)
                # reachable_from_shortest_lengths.append(total_len)

                sample_idx += 1

    return np.array(degrees_in_reachable), np.array(degrees_in_shortest), np.array(reachable_path_lengths), np.array(shortest_path_lengths), np.array(reachable_from_shortest_lengths), np.array(degrees_in_reachable_from_shortest)



def collect_shortest_paths(dataloader):
    shortest_path_lengths = []
    sample_idx = 0

    with torch.no_grad():
        for batch_inputs, _labels in tqdm(dataloader):
            for input in batch_inputs:
                input = np.array(input)
                maze = input[0]
                start_goal = input[-1]

                shortest_path = compute_shortest_path_mask(maze, start_goal)
                shortest_path_lengths.append(np.sum(shortest_path))

                sample_idx += 1

    return np.array(shortest_path_lengths)

def collect_euclidean_distances(dataloader):
    distances = []
    with torch.no_grad():
        for batch_inputs, _labels in tqdm(dataloader):
            for input in batch_inputs:
                start_goal = input[-1]

                sg01   = (start_goal != 0)
                start_goal = sg01[::2, ::2]
                rows, cols = np.where(start_goal)        # for 2D
                pts = np.column_stack((rows, cols))  # ≈ np.argwhere(a)

                if len(pts) != 2:
                    distance = 0
                else:
                    try:
                        distance = np.sqrt((pts[0][0] - pts[1][0])**2 + (pts[0][1] - pts[1][1])**2)
                    except:
                        plt.imshow(start_goal)
                        print(pts)
                        raise ValueError
                distances.append(distance)
    return distances

def scatter_with_fit(
    x, y, xlabel, ylabel, color_labels=None, title=None,
    remove_outliers=True, p=1.0,  # clip percent (1.0 => keep central 98%)
    point_labels=None             # new parameter for labeling points
):
    x = np.asarray(x, dtype=np.float64)
    y = np.asarray(y, dtype=np.float64)

    # Remove NaN/Inf
    m = np.isfinite(x) & np.isfinite(y)
    x, y = x[m], y[m]
    if color_labels is not None:
        color_labels = np.asarray(color_labels)[m]
    if point_labels is not None:
        point_labels = np.asarray(point_labels)[m]

    # Percentile-based outlier removal
    removed = 0
    if remove_outliers and len(x) > 5:
        x_lo, x_hi = np.percentile(x, [p, 100 - p])
        y_lo, y_hi = np.percentile(y, [p, 100 - p])
        m2 = (x >= x_lo) & (x <= x_hi) & (y >= y_lo) & (y <= y_hi)
        removed = np.count_nonzero(~m2)
        x, y = x[m2], y[m2]
        if color_labels is not None:
            color_labels = color_labels[m2]
        if point_labels is not None:
            point_labels = point_labels[m2]

    # Scatter
    if color_labels is not None:
        plt.scatter(x, y, c=color_labels, cmap="viridis", alpha=0.7)
    else:
        plt.scatter(x, y, alpha=0.7)

    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    if title:
        plt.title(title)

    # Add labels if provided
    if point_labels is not None:
        for xi, yi, label in zip(x, y, point_labels):
            plt.annotate(
                str(label),
                (xi, yi),
                textcoords="offset points",
                xytext=(5, 3),
                ha="left",
                fontsize=8
            )

    # Fit line
    if len(x) >= 2 and np.unique(x).size >= 2:
        slope, intercept = np.polyfit(x, y, 1)
        x_line = np.linspace(x.min(), x.max(), 200)
        y_line = slope * x_line + intercept

        y_pred = slope * x + intercept
        ss_res = np.sum((y - y_pred) ** 2)
        ss_tot = np.sum((y - y.mean()) ** 2)
        r2 = 1 - ss_res / ss_tot if ss_tot > 0 else np.nan

        plt.plot(
            x_line, y_line,
            label=f"Fit: y={slope:.3f}x+{intercept:.3f}, $R^2$={r2:.3f}",
            color="red"
        )
        plt.legend()

    plt.show()

import numpy as np
from collections import deque

def _neighbors4(i, j, H, W):
    if i+1 < H: yield i+1, j
    if i-1 >= 0: yield i-1, j
    if j+1 < W: yield i, j+1
    if j-1 >= 0: yield i, j-1

def _blocks_to_mask(blocks, shape):
    H, W = shape
    m = np.zeros((H, W), dtype=np.uint8)
    for group in blocks:
        for (i, j) in group:
            if 0 <= i < H and 0 <= j < W:
                m[i, j] = 1
    return m

def between_reachable_from_shortest(
    maze, start_goal, include_shortest_in_count=True
):
    """
    Multi-seed BFS from *all* shortest-path cells between the two 2×2 start blocks,
    treating the start blocks as walls. Returns (total_len, reach_mask, sp_mask).

    include_shortest_in_count:
        True  -> count shortest path cells + diversions
        False -> count diversions only (excludes shortest-path cells from the length)
    """
    H, W = maze.shape

    # 1) Find the two 2×2 start blocks
    blocks = get_start_coordinates(maze, start_goal)
    if len(blocks) != 2:
        return 0, np.zeros_like(maze, dtype=np.uint8), np.zeros_like(maze, dtype=np.uint8)

    # 2) Shortest-path cells between blocks
    sp_mask = compute_shortest_path_mask(maze, start_goal).astype(np.uint8)
    if sp_mask.sum() == 0:
        # No connection between blocks → nothing "between"
        return 0, np.zeros_like(maze, dtype=np.uint8), sp_mask

    # 3) Treat start blocks as walls
    blocks_mask = _blocks_to_mask(blocks, maze.shape)

    # 4) Allowed traversal: corridor cells that are NOT inside the start blocks
    allowed = (maze == 1) & (blocks_mask == 0)

    # 5) Multi-seed BFS from all shortest-path cells
    q = deque()
    visited = np.zeros((H, W), dtype=np.uint8)
    for (i, j) in zip(*np.where(sp_mask == 1)):
        if allowed[i, j]:
            visited[i, j] = 1
            q.append((i, j))

    while q:
        i, j = q.popleft()
        for ii, jj in _neighbors4(i, j, H, W):
            if allowed[ii, jj] and not visited[ii, jj]:
                visited[ii, jj] = 1
                q.append((ii, jj))

    reach_mask = visited  # all corridor cells reachable from anywhere on the shortest path, with starts blocked

    # 6) Length
    if include_shortest_in_count:
        total_len = int(reach_mask.sum())
    else:
        total_len = int((reach_mask & (sp_mask == 0)).sum())

    return total_len, reach_mask.astype(np.uint8), sp_mask


In [ ]:
def generate_maze_2px(
    H=48, W=48, num_branches=3, branch_length=4, min_gap_cells=1, rng=None
):
    assert H % 2 == 0 and W % 2 == 0
    rng = np.random.default_rng(rng)

    maze = np.zeros((H, W), dtype=np.uint8)
    start_goal = np.zeros((H, W), dtype=np.uint8)

    # center corridor (2 px tall)
    mid0, mid1 = H//2 - 1, H//2
    maze[mid0:mid1+1, 1:W-1] = 1

    # start blocks
    left_cols, right_cols = (2, 4), (W-4, W-2)
    start_goal[mid0:mid1+1, left_cols[0]:left_cols[1]]  = 1
    start_goal[mid0:mid1+1, right_cols[0]:right_cols[1]] = 1
    maze[mid0:mid1+1, left_cols[0]:left_cols[1]]  = 1
    maze[mid0:mid1+1, right_cols[0]:right_cols[1]] = 1

    def px_to_cell(c): return c // 2
    def cell_to_px(k): return 2 * k

    start_L_cell = (px_to_cell(left_cols[0]), px_to_cell(left_cols[1]-1))
    start_R_cell = (px_to_cell(right_cols[0]), px_to_cell(right_cols[1]-1))

    cell_W = W // 2
    candidates_cells = list(range(1, cell_W - 1))  # avoid border cells

    # ban cells overlapping/touching starts by min_gap_cells
    banned = set()
    for a, b in (start_L_cell, start_R_cell):
        for k in range(a - min_gap_cells, b + 1 + min_gap_cells):
            banned.add(k)
    candidates_cells = [k for k in candidates_cells if k not in banned]

    rng.shuffle(candidates_cells)

    chosen_cells = []
    for k in candidates_cells:
        if len(chosen_cells) >= num_branches:
            break
        # keep at least min_gap_cells between branches
        if all(abs(k - kk) >= (1 + min_gap_cells) for kk in chosen_cells):
            chosen_cells.append(k)

    pix_len = 2 * int(branch_length)
    branches = []
    for k in chosen_cells:
        c0, c1 = cell_to_px(k), cell_to_px(k) + 2  # [c0:c1)
        can_up = (mid0 - pix_len) >= 1
        can_dn = (mid1 + 1 + pix_len) <= (H - 1)   # exclusive end ≤ H-1 (keep border)

        # pick uniformly among feasible directions
        options = []
        if can_up: options.append('up')
        if can_dn: options.append('down')
        if not options:
            continue
        direction = rng.choice(options)

        if direction == 'up':
            r0, r1 = mid0 - pix_len, mid0        # [r0:r1)
        else:
            r0, r1 = mid1 + 1, mid1 + 1 + pix_len

        maze[r0:r1, c0:c1] = 1
        branches.append({'c0': c0, 'c1': c1-1, 'direction': direction, 'r0': r0, 'r1': r1-1})

    return maze.astype(np.uint8), start_goal.astype(np.uint8), branches


In [ ]:
def circular_com_1d(weights, L):
    """Circular center-of-mass over a 1D ring of length L."""
    # indices -> angles on [0, 2π)
    idx = np.arange(L)
    theta = 2 * np.pi * idx / L
    w = weights.astype(float)
    wsum = w.sum()
    if wsum == 0:
        return np.nan  # or L/2 as a fallback
    C = (w * np.cos(theta)).sum() / wsum
    S = (w * np.sin(theta)).sum() / wsum
    theta_bar = np.arctan2(S, C) % (2 * np.pi)
    return (theta_bar / (2 * np.pi)) * L  # in index units [0, L)

def toroidal_com(frame):
    """
    frame: (H, W) intensity map on a torus.
    Returns (y_com, x_com) in index units with periodic CoM.
    """
    H, W = frame.shape
    # marginals along each axis
    wy = frame.sum(axis=1)   # length H
    wx = frame.sum(axis=0)   # length W
    y_com = circular_com_1d(wy, H)
    x_com = circular_com_1d(wx, W)
    return np.array([y_com, x_com], dtype=float)

def wrap_min_signed(delta_raw, L):
    """Minimal signed displacement on a ring of length L."""
    return ((delta_raw + L/2) % L) - L/2

def toroidal_trajectory(tensor):
    """
    tensor: (T, H, W)
    Returns CoM per frame with toroidal CoM: positions shape (T, 2) as (y, x).
    """
    T, H, W = tensor.shape
    pos = np.empty((T, 2), dtype=float)
    for t in range(T):
        pos[t] = toroidal_com(tensor[t])
    return pos  # CoM in index coords

def toroidal_net_displacement(tensor):
    """
    Robust net displacement integrating minimal periodic steps.
    Returns: net_disp (Δy, Δx), unit_direction (2,), and per-step displacements.
    """
    T, H, W = tensor.shape
    pos = toroidal_trajectory(tensor)  # (T, 2)
    d_raw = np.diff(pos, axis=0)       # (T-1, 2)
    dy = wrap_min_signed(d_raw[:, 0], H)
    dx = wrap_min_signed(d_raw[:, 1], W)
    disp_steps = np.stack([dy, dx], axis=1)
    net_disp = disp_steps.sum(axis=0)
    norm = np.linalg.norm(net_disp)
    unit_dir = net_disp / norm if norm > 0 else np.array([np.nan, np.nan])
    return net_disp, unit_dir, disp_steps


# Loading Models and Data

## Loading Models

In [ ]:
## CORRELATED DOTS ##

wandb_name, checkpoint = "fancy-silence-24", 370 #BIOPLNN
# wandb_name, checkpoint = "glowing-deluge-31", 330 #CNN
wandb_name, checkpoint = "rosy-morning-40", 390 #Best, max_speed=1
# wandb_name, checkpoint = "eager-cosmos-37", 390 #Best, max_speed=5

## MAZES ##

# wandb_name, checkpoint = "dauntless-elevator-522", 390 # ReLU, with I->I recurrence

# ——— Load model config & instantiate ———
model, classifier, num_steps, cfg, state_dict = load_model_and_config(wandb_name, checkpoint_path)

model.eval()
model.to(device)

classifier.eval()
classifier.to(device)

## Get different variations on one maze

In [ ]:
batch_size = 1
train_loader, test_loader = initialize_dataloader(
    seed=42, root=maze_data_path, batch_size=batch_size, dataset="mazes"
)
all_samples = list(train_loader)

In [ ]:
inputs = []
for idx in range(10,11):
    inputs.append(get_maze_variations(all_samples, idx=idx, num_samples=5000))
inputs = torch.concat(inputs, dim=0)

cut_out_infs = False

distances = []
used_inputs = []
for i, input in enumerate(inputs):
    distance = get_dist(np.array(input[0]), np.array(input[-1]))
    if distance != np.inf and cut_out_infs:
        distances.append(distance)
        used_inputs.append(input) 
    else:
        distances.append(distance)
        used_inputs.append(input)  

inputs_tensor = torch.stack(used_inputs, dim=0)
num_samples = len(inputs_tensor)
dummy_targets = torch.zeros(inputs_tensor.shape[0])  # Placeholder targets

maze_variations_dataset = TensorDataset(inputs_tensor, dummy_targets)
train_loader = DataLoader(maze_variations_dataset, batch_size=512, shuffle=False)
decision_step = 20

train_ea, train_ia, train_oa, train_decisions, trainloader, final_decisions = get_activations(model, classifier, num_samples, train_loader, decision_step, num_steps=num_steps)

## Get a positive and negative maze pairing

In [ ]:
positive = torch.clone(inputs_tensor[0])
temp_positive = torch.zeros_like(positive[0])
temp_positive[15:17, 15:17] = 1
temp_positive[21:23, 11:13] = 1
plt.imshow(temp_positive + positive[0])
positive[-1] = temp_positive

negative = torch.clone(positive)
negative[0:3, 19:21, 19:21] = 0
input_tensor = torch.stack((positive, negative))

dummy_targets = torch.zeros(input_tensor.shape[0])  # Placeholder targets
posnegset = TensorDataset(input_tensor, dummy_targets)
loader = DataLoader(posnegset, batch_size=512, shuffle=False)
decision_step = 20
train_ea, train_ia, train_oa, train_decisions, trainloader, final_decisions = get_activations(model, classifier, 2, loader, decision_step, num_steps=num_steps)

In [ ]:
plt.imshow(input_tensor[0, 2] + input_tensor[0, -1]*2)

## Time v difficulty

### Get time to decisions for various mazes

### Regular

In [ ]:
loader, test_loader = initialize_dataloader(
    seed=42, root=maze_data_path, batch_size=256, dataset="mazes", annotation_scaling=3
)

num_samples = 10000

all_indices = np.arange(len(loader.dataset))
subset_indices = np.random.choice(all_indices, size=num_samples, replace=False)
decision_step = 20

subset_dataset = Subset(loader.dataset, subset_indices)

# mirror original loader settings where possible
loader = DataLoader(
    subset_dataset,
    batch_size=loader.batch_size,
    shuffle=False,  # keep consistent ordering for preallocation fill
    num_workers=getattr(loader, "num_workers", 0),
    pin_memory=getattr(loader, "pin_memory", False),
    drop_last=False
)

### Branch mazes

In [ ]:
random_seed = np.arange(1000)
branch_lengths = [1, 2, 3, 4, 5, 6, 7, 8]
num_branches = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
dataset = []
for seed in random_seed:    
    branch_cell_counts = []
    for length in branch_lengths:
        for n_branches in num_branches:
            maze, start_goal, branches = generate_maze_2px(
                H=48, W=48, num_branches=n_branches, branch_length=length, rng=seed
            )
            stacked = np.stack([maze, maze, maze, start_goal], axis=0).astype(np.float32)
            dataset.append((stacked, 1))   # <- add label here
            branch_cell_counts.append(length * n_branches)

# now each element is (sample, label)
loader = DataLoader(
    dataset,
    batch_size=128,
    shuffle=False,
    drop_last=False
)

In [ ]:
final_decisions = get_decisions(classifier, loader, decision_step)
decision_points = np.zeros(len(loader.dataset)) + 20
for step in range(20, 0, -1):
    decisions = get_decisions(classifier, loader, step)
    for input_idx, decision in enumerate(decisions):
        if decision == final_decisions[input_idx]:
            decision_points[input_idx] = step

In [ ]:
final_decisions_matrix = final_decisions.reshape((len(random_seed), len(branch_lengths), len(num_branches)))
mask = final_decisions_matrix > 0
decision_points_matrix = decision_points.reshape((len(random_seed), len(branch_lengths), len(num_branches)))

In [ ]:
decision_points_filled = np.where(mask, decision_points_matrix, 20)
decision_points_avg = decision_points_filled.mean(axis=0)

In [ ]:
mask.sum(axis=0)

In [ ]:
# mask: same shape as final_decisions_matrix, True = ignore
masked = np.ma.array(decision_points_matrix, mask=~mask)

# take mean along the seed axis (say axis=0), keep shape of the rest
decision_points_avg = masked.mean(axis=0).filled(np.nan)

In [ ]:
decision_points_avg = masked.mean(axis=0).filled(np.nan).flatten()
plt.scatter(branch_cell_counts, decision_points_avg)
plt.xlabel("n_branch_cells")
plt.ylabel("steps to decision")
plt.show()

In [ ]:
plt.imshow(decision_points_avg, origin="lower")
plt.xlabel("num_branches")
plt.ylabel("branch_length")
plt.colorbar(label="Steps until decision")
plt.show()

In [ ]:
L = len(branch_lengths)
B = len(num_branches)

# shapes: [seeds, L, B]
D = decision_points_matrix
M = final_decisions_matrix > 0  # success mask

# mask failures as NaN so they don't skew means
D_masked = np.where(M, D, np.nan)

# stats per (length, branches)
mean_steps = np.nanmean(D_masked, axis=0)              # [L,B]
std_steps  = np.nanstd(D_masked,  axis=0)
succ_rate  = np.mean(M, axis=0)                        # [L,B] in [0,1]


In [ ]:
# vary n_branches, different lines for lengths
plt.figure()
for i, Lval in enumerate(branch_lengths):
    plt.plot(num_branches, mean_steps[i,:], marker="o", label=f"len={Lval}")
plt.xlabel("# branches"); plt.ylabel("steps")
plt.title("dp vs #branches (lines = branch lengths)")
plt.legend(ncol=2, fontsize=8)
plt.tight_layout()

# vary branch_length, different lines for branches
plt.figure()
for j, Bval in enumerate(num_branches):
    plt.plot(branch_lengths, mean_steps[:,j], marker="o", label=f"branches={Bval}")
plt.xlabel("branch length"); plt.ylabel("steps")
plt.title("dp vs branch length (lines = #branches)")
plt.legend(ncol=3, fontsize=8)
plt.tight_layout()


### Get difficulty marker -- degrees in shortest or reachable path and compare with time to decision

In [ ]:
all_shortest_path_lengths = collect_shortest_paths(loader)

In [ ]:
values, counts = np.unique(all_shortest_path_lengths, return_counts=True)
# sort counts descending
order = np.argsort(-counts)
second_most_common = values[order[:10]]
second_count = counts[order[:10]]
indices = np.where(all_shortest_path_lengths == 140)[0]

# mirror original loader settings where possible
loader = DataLoader(
    [subset_dataset[i] for i in indices],
    batch_size=loader.batch_size,
    shuffle=False,  # keep consistent ordering for preallocation fill
    num_workers=getattr(loader, "num_workers", 0),
    pin_memory=getattr(loader, "pin_memory", False),
    drop_last=False
)

In [ ]:
data = [loader.dataset[i] for i in range(len(mask)) if mask[i]]

In [ ]:
plt.imshow(data[index][0][0][::2, ::2])

In [ ]:
indices[0]

In [ ]:
index = indices[15][0]
plt.imshow(data[index][0][0] + 2*data[index][0][-1])
plt.show()

print(decision_points[mask][index])
print(shortest_path_lengths[mask][index])
print(degrees_in_shortest[mask][index])

# print(degrees_in_reachable[mask][index])
print(reachable_path_lengths[mask][index])
# print(degrees_in_shortest_path[mask][index])

In [ ]:
degrees_in_reachable, degrees_in_shortest, reachable_path_lengths, shortest_path_lengths, reachable_from_shortest_lengths, degrees_in_reachable_from_shortest = collect_degrees(loader)

In [ ]:
euclidean_distances = collect_euclidean_distances(loader)

In [ ]:
len(degrees_in_reachable)

In [ ]:
final_decisions = get_decisions(classifier, loader, decision_step)
decision_points = np.zeros(len(loader.dataset)) + 20
for step in range(20, 0, -1):
    decisions = get_decisions(classifier, loader, step)
    for input_idx, decision in enumerate(decisions):
        if decision == final_decisions[input_idx]:
            decision_points[input_idx] = step

In [ ]:
mask1 = final_decisions > 0
mask2 = shortest_path_lengths > 4
mask3 = reachable_from_shortest_lengths > 0
mask4 = decision_points > 3
mask = mask1 & mask2
# dps_normalized = decision_points[mask] / shortest_path_lengths[mask]
# degrees_shortest_normalized = degrees_in_shortest[mask] / shortest_path_lengths[mask]
# degrees_reachable_normalized = degrees_in_reachable[mask] / reachable_path_lengths[mask]
# degrees_shortest_reachable_normalized = degrees_in_reachable_from_shortest[mask] / reachable_from_shortest_lengths[mask]

In [ ]:
indices = np.argwhere(decision_points[mask] == 3)

In [ ]:
plt.scatter(np.array(euclidean_distances)[mask], decision_points[mask])
plt.title("Time to Decision vs Euclidean Distance", fontsize=14)
plt.ylabel("Time to Decision", fontsize=12)
plt.xlabel("Euclidean Distance Between Start and Goal", fontsize=12)
#add a line of best fit
plt.plot(np.unique(np.array(euclidean_distances)[mask]), np.poly1d(np.polyfit(np.array(euclidean_distances)[mask], decision_points[mask], 1))(np.unique(np.array(euclidean_distances)[mask])), color='red')
# add r-squared value
r_squared = np.corrcoef(np.array(euclidean_distances)[mask], decision_points[mask])[0,1]**2
plt.legend([f"R-squared: {r_squared:.2f}"], loc="best", fontsize=12)

In [ ]:
plt.scatter(shortest_path_lengths[mask], decision_points[mask])
plt.title("Time to Decision vs Geodesic Distance", fontsize=14)
plt.ylabel("Time to Decision", fontsize=12)
plt.xlabel("Geodesic Distance Between Start and Goal", fontsize=12)
#add a line of best fit 
plt.plot(np.unique(np.array(shortest_path_lengths)[mask]), np.poly1d(np.polyfit(np.array(shortest_path_lengths)[mask], decision_points[mask], 1))(np.unique(np.array(shortest_path_lengths)[mask])), color='red')
# add r-squared value
r_squared = np.corrcoef(np.array(shortest_path_lengths)[mask], decision_points[mask])[0,1]**2
plt.legend([f"R-squared: {r_squared:.2f}"], loc="best", fontsize=12)


In [ ]:
degrees_in_reachable, degrees_in_shortest, reachable_path_lengths, shortest_path_lengths = degrees_in_reachable[mask], degrees_in_shortest[mask], reachable_path_lengths[mask], shortest_path_lengths[mask]

In [ ]:
len(reachable_from_shortest_lengths)

In [ ]:
import numpy as np
import statsmodels.api as sm

def run_incremental(y, controls, newvar, control_names, newvar_name):
    """
    Fit baseline (controls) vs full (controls + newvar).
    Return baseline R², full R², ΔR², coef and p-value for newvar.
    """
    # Baseline
    Xc = sm.add_constant(np.column_stack(controls))
    model_c = sm.OLS(y, Xc).fit()
    R2_c = model_c.rsquared

    # Full
    Xf = sm.add_constant(np.column_stack(controls + [newvar]))
    model_f = sm.OLS(y, Xf).fit()
    R2_f = model_f.rsquared

    coef = model_f.params[-1]
    pval = model_f.pvalues[-1]

    return {
        "baseline_R2": R2_c,
        "full_R2": R2_f,
        "delta_R2": R2_f - R2_c,
        "coef": coef,
        "pval": pval
    }

results = {}

# 1. Each variable on top of shortest_path_lengths
extra_vars = {
    "reachable_path_lengths": reachable_path_lengths,
    "degrees_in_reachable": degrees_in_reachable,
    "degrees_in_shortest": degrees_in_shortest,
}
for name, arr in extra_vars.items():
    res = run_incremental(
        decision_points[mask],
        [shortest_path_lengths],
        arr,
        ["shortest_path_lengths"],
        name
    )
    results[f"{name} | shortest_path_lengths"] = res

# 2. Each degrees var on top of shortest_path_lengths + corresponding lengths
pairs = [
    ("degrees_in_reachable", degrees_in_reachable,
     "reachable_path_lengths", reachable_path_lengths),
    ("degrees_in_shortest", degrees_in_shortest,
     "shortest_path_lengths", shortest_path_lengths),
]
for dname, darr, lname, larr in pairs:
    res = run_incremental(
        decision_points[mask],
        [shortest_path_lengths, larr],
        darr,
        ["shortest_path_lengths", lname],
        dname
    )
    results[f"{dname} | shortest_path_lengths+{lname}"] = res

# 3. Each degrees var on top of shortest_path_lengths + its length + degrees_in_shortest
for dname, darr, lname, larr in pairs:
    res = run_incremental(
        decision_points[mask],
        [shortest_path_lengths, larr, degrees_in_shortest],
        darr,
        ["shortest_path_lengths", lname, "degrees_in_shortest"],
        dname
    )
    results[f"{dname} | shortest_path_lengths+{lname}+degrees_in_shortest"] = res

# ----------------------------------------------------------------------
# Print results
for key, val in results.items():
    print(f"\n=== {key} ===")
    print(f"baseline R² = {val['baseline_R2']:.3f}")
    print(f"full R²     = {val['full_R2']:.3f}")
    print(f"ΔR²         = {val['delta_R2']:.3f}")
    print(f"coef        = {val['coef']:.4f}")
    print(f"pval        = {val['pval']:.2e}")


In [ ]:
data = [loader.dataset[i] for i in range(len(loader.dataset)) if mask[i] == True]

In [ ]:
min_idx = np.argsort(degrees_shortest_normalized)[:10]
max_idx = np.argsort(degrees_shortest_normalized)[-10:]

# Target values
target_x = 3.02
target_y = 0.026

# Target values
# target_x = 3.03
# target_y = 0.027

# Use np.isclose in case of float rounding
same_mask = (np.isclose(degrees_shortest_normalized, target_x) &
        np.isclose(dps_normalized, target_y))

indices = np.where(same_mask)[0]


In [ ]:
index=10
plt.imshow(data[index][0][0].numpy() + 2*data[index][0][-1].numpy() + 3*compute_shortest_path_mask(data[index][0][0].numpy(), data[index][0][-1].numpy()))
plt.show()

total_len, reach_mask, sp_mask = between_reachable_from_shortest(data[index][0][0].numpy(), data[index][0][-1].numpy(), include_shortest_in_count=False)
print("Total reachable (shortest path + diversions):", total_len)

In [ ]:
for min in indices1:
    plt.imshow(data[min][0][0] + 2*data[min][0][-1])
    plt.title(f"avg deg in shortest path:  {degrees_shortest_normalized[min]}")
    plt.show()

## Basic dataloading

In [ ]:
batch_size = 128
num_train_samples = batch_size * 40
num_test_samples = batch_size * 1

# MAZES

# dim = 48
# num_steps = 20
# decision_step = 20
# train_loader, test_loader = initialize_dataloader(
#     seed=42, root=maze_data_path, batch_size=batch_size, dataset="mazes"
# )

# DOTS

n_frames = 30
resolution = (128, 128)
correlation = (0.25,0.75)
max_speed = 1
train_loader, test_loader = initialize_dataloader(
        seed=12, dataset="correlated_dots",
        resolution=resolution, n_frames=n_frames, correlation=0.25, max_speed=max_speed, samples_per_epoch=num_train_samples,
        batch_size=batch_size)
decision_step = n_frames
num_steps = n_frames
dim = resolution[0]

# # # DOTS GOING ONE DIRECTION

#N, NE 2, 4
# N, S 2, 3
# N, W 2, 1

label_to_keep = [0,1]
train_loader = keep_only_label_from_loader(train_loader, label_to_keep)
num_train_samples = len(train_loader.dataset)
test_loader = keep_only_label_from_loader(test_loader, label_to_keep)
num_test_samples = len(test_loader.dataset)

# # ADD A WORKING MEMORY PERIOD
# n_extra_frames = 20

# # Add a blank period to every sample in train_loader and test_loader
# def add_blank_to_loader(loader, n_extra_frames, dim):
#     # This function returns a new loader with blank frames appended to each sample
#     from torch.utils.data import DataLoader, TensorDataset

#     all_inputs = []
#     all_labels = []
#     for batch in loader:
#         # batch[0]: (batch_size, n_frames, dim, dim)
#         # batch[1]: labels
#         inputs, labels = batch
#         # Add blank period to each sample in the batch
#         # blank_period: (n_extra_frames, dim, dim)
#         blank = torch.zeros((inputs.shape[0], n_extra_frames, inputs.shape[2], dim, dim), dtype=inputs.dtype, device=inputs.device)
#         inputs_with_blank = torch.cat([inputs, blank], dim=1)
#         all_inputs.append(inputs_with_blank)
#         all_labels.append(labels)
#     all_inputs = torch.cat(all_inputs, dim=0)
#     all_labels = torch.cat(all_labels, dim=0)
#     new_dataset = TensorDataset(all_inputs, all_labels)
#     new_loader = DataLoader(new_dataset, batch_size=loader.batch_size, shuffle=False)
#     return new_loader

# train_loader = add_blank_to_loader(train_loader, n_extra_frames, dim)
# test_loader = add_blank_to_loader(test_loader, n_extra_frames, dim)
# num_steps = n_frames + n_extra_frames
# decision_step = num_steps

### Selecting specific samples

In [ ]:
dataset = train_loader.dataset
batch_size = train_loader.batch_size if hasattr(train_loader, 'batch_size') and train_loader.batch_size is not None else 64

# Get total number of samples
N = len(dataset)
labels = torch.tensor([dataset[i][1] for i in range(N)])

# First, get the final decisions for all samples at full length
final_decisions = torch.zeros(N, dtype=torch.long)
for start in tqdm(range(0, N, batch_size)):
    end = min(start + batch_size, N)
    batch_inputs = [dataset[i][0][:num_steps] for i in range(start, end)]
    batch_inputs = torch.stack(batch_inputs).to(device)
    output = classifier(batch_inputs)
    decisions = output.argmax(dim=1).cpu()
    final_decisions[start:end] = decisions

# Find indices where the final decision is 0 or 1
keep_mask = (final_decisions == 2) | (final_decisions == 4)
keep_indices = torch.where(keep_mask)[0]

# Now, apply additional constraints:
# - Keep all incorrect examples (final_decision != label)
# - Keep only 100 correct examples (final_decision == label)

final_decisions_kept = final_decisions[keep_indices]
labels_kept = labels[keep_indices]

correct_mask = (final_decisions_kept == labels_kept)
incorrect_mask = ~correct_mask

incorrect_indices = keep_indices[incorrect_mask]
correct_indices = keep_indices[correct_mask]

# Randomly select up to 100 correct and up to 100 incorrect examples (if there are at least 100 of each)
num_correct_to_keep = min(100, len(correct_indices))
num_incorrect_to_keep = min(100, len(incorrect_indices))

if num_correct_to_keep > 0:
    perm_correct = torch.randperm(len(correct_indices))[:num_correct_to_keep]
    selected_correct_indices = correct_indices[perm_correct]
else:
    selected_correct_indices = torch.tensor([], dtype=torch.long)

if num_incorrect_to_keep > 0:
    perm_incorrect = torch.randperm(len(incorrect_indices))[:num_incorrect_to_keep]
    selected_incorrect_indices = incorrect_indices[perm_incorrect]
else:
    selected_incorrect_indices = torch.tensor([], dtype=torch.long)

# Combine selected correct and incorrect indices
final_keep_indices = torch.cat([selected_incorrect_indices, selected_correct_indices])

# Sort indices for consistency (optional)
final_keep_indices = final_keep_indices.sort().values
keep_indices = final_keep_indices

inputs = torch.stack([dataset[i][0] for i in keep_indices])
labels = torch.tensor([dataset[i][1] for i in keep_indices])
dataset = TensorDataset(inputs, labels)
N = len(dataset)

# Now, record decisions over all steps for just these samples
decisions_matrix = torch.zeros((N, num_steps), dtype=torch.long)
for idx, input_length in tqdm(enumerate(range(1, num_steps+1))):
    for start in range(0, N, batch_size):
        end = min(start + batch_size, N)
        batch_inputs = [dataset[i][0][:input_length] for i in range(start, end)]
        batch_inputs = torch.stack(batch_inputs).to(device)
        output = classifier(batch_inputs)
        decisions = output.argmax(dim=1).cpu()
        decisions_matrix[start:end, idx] = decisions

In [ ]:
incorrect_indices

In [ ]:
# Now, for each sample, plot if the final decision is incorrect
for i in range(N):
    sample_decisions = decisions_matrix[i].tolist()
    label = labels[i].item()
    if (label == sample_decisions[-1]) and (label == 4):
        plt.plot(sample_decisions)
        plt.title(f"input {i}, label: {label}, final_decision: {sample_decisions[-1]}")
        plt.show()
    if i > 100:
        break

In [ ]:
# Indices for each group
indices_2_early_correct = [61]
indices_2_early_incorrect = [3]
indices_2_late_correct = [15]
indices_2_late_incorrect = [2]

indices_4_early_correct = [49]
indices_4_early_incorrect = [50]
indices_4_late_correct = [30]
indices_4_late_incorrect = [33]

# Combine all indices and keep track of groupings
all_indices = (
    indices_2_early_correct +
    indices_2_early_incorrect +
    indices_2_late_correct +
    indices_2_late_incorrect +
    indices_4_early_correct +
    indices_4_early_incorrect +
    indices_4_late_correct +
    indices_4_late_incorrect
)

# Early/Late and label for each index, in the same order as all_indices

# The order in all_indices is:
# [indices_2_early_correct, indices_2_early_incorrect, indices_2_late_correct, indices_2_late_incorrect,
#  indices_4_early_correct, indices_4_early_incorrect, indices_4_late_correct, indices_4_late_incorrect]
# So, we need to assign 'early' or 'late' for each group in that order.

early_late_labels = (
    ['early'] * len(indices_2_early_correct) +
    ['early'] * len(indices_2_early_incorrect) +
    ['late'] * len(indices_2_late_correct) +
    ['late'] * len(indices_2_late_incorrect) +
    ['early'] * len(indices_4_early_correct) +
    ['early'] * len(indices_4_early_incorrect) +
    ['late'] * len(indices_4_late_correct) +
    ['late'] * len(indices_4_late_incorrect)
)

# Similarly, assign class labels for each group in that order.
class_labels = (
    [0] * (len(indices_2_early_correct) + len(indices_2_early_incorrect) + len(indices_2_late_correct) + len(indices_2_late_incorrect)) +
    [1] * (len(indices_4_early_correct) + len(indices_4_early_incorrect) + len(indices_4_late_correct) + len(indices_4_late_incorrect))
)

# Get tensors for inputs, labels, decisions, and early/late
inputs_tensor = torch.stack([dataset[i][0] for i in all_indices])
labels_tensor = torch.tensor([labels[i] for i in all_indices])
decisions_tensor = torch.tensor([decisions_matrix[i, -1] for i in all_indices])
early_late_tensor = torch.tensor([0 if lbl == 'early' else 1 for lbl in early_late_labels])  # 0: early, 1: late

# Optionally, print to verify
print("inputs_tensor shape:", inputs_tensor.shape)
print("labels_tensor:", labels_tensor)
print("decisions_tensor:", decisions_tensor)
print("early_late_tensor:", early_late_tensor)

# Create a TensorDataset with the selected inputs and labels
subset_dataset = TensorDataset(inputs_tensor, labels_tensor)

# Create a DataLoader from the subset dataset
train_loader = DataLoader(subset_dataset, batch_size=64, shuffle=False)
num_train_samples = len(train_loader.dataset)


In [ ]:
#save these to file
torch.save(train_loader, "train_loader.pt")
torch.save(labels_tensor, "labels_tensor.pt")
torch.save(decisions_tensor, "decisions_tensor.pt")
torch.save(early_late_tensor, "early_late_tensor.pt")

In [ ]:
train_loader = torch.load("train_loader.pt")
labels_tensor = torch.load("labels_tensor.pt")
decisions_tensor = torch.load("decisions_tensor.pt")
early_late_tensor = torch.load("early_late_tensor.pt")
num_train_samples = len(train_loader.dataset)

# PCA

In [ ]:
train_ea, train_ia, train_oa, train_decisions, train_inputs, train_labels = get_activations(model, classifier, num_train_samples, train_loader, decision_step, num_steps=num_steps,dim=dim)
test_ea, test_ia, test_oa, test_decisions, test_inputs, test_labels = get_activations(model, classifier, 200, test_loader, decision_step, num_steps=num_steps,dim=dim)

# CHOOSE CORRECT EXAMPLES ONLY
only_correct = False
if only_correct:
    save_idx_train = np.flatnonzero(np.squeeze(train_decisions) == np.squeeze(train_labels))
    train_ea, train_decisions, train_labels, train_inputs = [np.asarray(a)[save_idx_train] for a in (train_ea, train_decisions, train_labels, train_inputs)]
    save_idx_test = np.flatnonzero(np.squeeze(test_decisions) == np.squeeze(test_labels))
    test_ea, test_decisions, test_labels, test_inputs = [np.asarray(a)[save_idx_test] for a in (test_ea, test_decisions, test_labels, test_inputs)]

# # CHOOSE ONLY EXAMPLES THAT ARE CLASSIFIED AS ONE OF THE LABELS IN label_to_keep
idx = np.flatnonzero(np.isin(train_decisions, label_to_keep))
train_ea, train_decisions, train_labels = [np.asarray(a)[idx] for a in (train_ea, train_decisions, train_labels)]
idx = np.flatnonzero(np.isin(test_decisions, label_to_keep))
test_ea, test_decisions, test_labels = [np.asarray(a)[idx] for a in (test_ea, test_decisions, test_labels)]

In [ ]:
incorrect_decisions = 0
for i in range(len(train_decisions)):
    if train_decisions[i] != train_labels[i]:
        incorrect_decisions += 1

print(f"Incorrect decisions: {incorrect_decisions/len(train_decisions)}")

## Record Integrated Directions

In [ ]:
def get_integrated_directions(train_inputs, train_labels, num_steps):
    num_intg_examples = len(train_inputs)
    integrated_directions = np.zeros((num_intg_examples, num_steps, 2))
    # Get all indices in the train set
    all_indices = np.arange(len(train_inputs))

    # Sample without replacement
    subset_indices = np.random.choice(all_indices, size=num_intg_examples, replace=False)
    # Sort indices
    subset_indices = np.sort(subset_indices)

    with torch.no_grad():
        for idx, sample_idx in enumerate(subset_indices):
            sample, label = train_inputs[sample_idx], train_labels[sample_idx]
            sample = np.array(sample)
            for i in range(num_steps):
                net_disp, unit_dir, disp_steps = toroidal_net_displacement(sample[:i+1, 0])
                integrated_directions[idx, i] = net_disp
    return integrated_directions, subset_indices

## Extract Activations Regularly

In [ ]:
start_idx = 0
end_idx = decision_step

#TRAIN
act = train_ea[:, start_idx:end_idx]
pca_acts = act.reshape(*act.shape[:-3], -1)
decisions = train_decisions

#TEST
# act = test_ea[:, start_idx:end_idx]
# pca_acts = act.reshape(*act.shape[:-3], -1)
# decisions = test_decisions

## Get Output Space

In [ ]:
# TAKE OUTPUT ACTIVATIONS AND TURN THEM INTO DECISIONS SPACE ACTIVATIONS
conv_w = state_dict["rnn.areas.0.out_convs.1->out.0.weight"] #torch.Size([8, 8, 5, 5])
conv_b = state_dict["rnn.areas.0.out_convs.1->out.0.bias"] #torch.Size([8])
lin_w = state_dict["readout.1.weight"] #torch.Size([8, 512])
lin_b = state_dict["readout.1.bias"] #torch.Size([512])

pca_acts = batched_readout_from_extracted(torch.tensor(train_oa), conv_w, conv_b, lin_w, lin_b, use_relu=True, pool="max")
decisions = train_decisions

## Get Decision Space by Training Decoder

In [ ]:
#inputs

train_steps = np.arange(30)
# train_steps = np.array([39])

act = train_ea[:, train_steps]
act = act.reshape(*act.shape[:-3], -1)
act = act.reshape(-1, act.shape[-1])

test_act = test_ea[:, train_steps]
test_act = test_act.reshape(*test_act.shape[:-3], -1)
test_act = test_act.reshape(-1, test_act.shape[-1])

#labels - decisions

train_dec_repeated = torch.tensor(train_decisions).repeat_interleave(len(train_steps))
test_dec_repeated = torch.tensor(test_decisions).repeat_interleave(len(train_steps))
decoder_train_labels = train_dec_repeated
decoder_test_labels = test_dec_repeated

#labels - intg directions

# train_intg_directions, train_intg_indices = get_integrated_directions(train_inputs, train_labels, num_steps)
# all_decoder_train_labels = torch.tensor(train_intg_directions[:, train_steps].reshape(-1, 2))
# test_intg_directions, test_intg_indices = get_integrated_directions(test_inputs, test_labels, num_steps)
# all_decoder_test_labels = torch.tensor(test_intg_directions[:, train_steps].reshape(-1, 2))
# # for now, just take x
# decoder_train_labels = all_decoder_train_labels[:, 1]
# decoder_test_labels = all_decoder_test_labels[:, 1]

In [ ]:
print(torch.tensor(act).shape,
    decoder_train_labels.shape,
    torch.tensor(test_act).shape,
    decoder_test_labels.shape)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.decomposition import PCA

# ---------- shape helpers ----------
def ensure_time_axis(X):
    X = np.asarray(X)
    if X.ndim == 2:  # (N, D) -> (N, 1, D)
        X = X[:, None, :]
    if X.ndim != 3:
        raise ValueError("X must be (N,T,D) or (N,D).")
    return X

# ---------- 1) train linear decoder ----------
def train_linear_decoder(X, y, C=1.0, max_iter=5000):
    X = ensure_time_axis(X)             # (N,T,D)
    H_T = X[:, -1, :]                   # final-time features (N,D)
    clf = LogisticRegression(C=C, solver="lbfgs", max_iter=max_iter)
    clf.fit(H_T, y)
    w = clf.coef_.ravel().astype(float) # (D,)
    d = w / (np.linalg.norm(w) + 1e-12) # unit decision axis
    return clf, d

# ---------- 2) project onto d & get residuals in d⊥ ----------
def project_to_d_and_residual(X, d):
    X = ensure_time_axis(X)             # (N,T,D)
    if X.shape[-1] != d.size:
        raise ValueError(f"Feature dim mismatch: X has D={X.shape[-1]}, d has {d.size}.")
    # projection along d
    Dcoord = np.einsum('ntd,d->nt', X, d)             # (N,T)
    # subtract component along d -> residuals in d⊥
    X_parallel = Dcoord[..., None] * d[None, None, :] # (N,T,D)
    R = X - X_parallel                                # (N,T,D)
    return Dcoord, R

# ---------- 3) PCA in d⊥ to get top orthogonal axis z ----------
def pca_in_d_perp(R, d):
    N, T, D = R.shape
    R_flat = R.reshape(N*T, D)
    pca = PCA(n_components=1)
    pca.fit(R_flat)
    z = pca.components_[0]                  # (D,)
    # Gram–Schmidt to ensure z ⟂ d
    z = z - (z @ d) * d
    z /= (np.linalg.norm(z) + 1e-12)
    return z, pca

# ---------- 4) project trajectories into (d, z) ----------
def project_to_plane(X, d, z):
    X = ensure_time_axis(X)                 # (N,T,D)
    Dcoord = np.einsum('ntd,d->nt', X, d)   # (N,T)
    Zcoord = np.einsum('ntd,d->nt', X, z)   # (N,T)
    return Dcoord, Zcoord

# ---------- 5) plot single-trial trajectories colored by decision ----------
def plot_trajectories_in_plane(X, y, d, z, title="Trajectories in (d, z) plane", n_traces=20):
    """
    Plot single-trial trajectories in the (d, z) plane, showing the same number of single-trial traces per class.

    Args:
        X: array-like, shape (N, T, D) or (N, D)
        y: array-like, shape (N,)
        d: array-like, shape (D,)
        z: array-like, shape (D,)
        title: str
        n_traces: int, number of single-trial traces to plot per class (default: 20)
    """
    Dcoord, Zcoord = project_to_plane(X, d, z)  # (N,T)
    y = np.asarray(y)
    classes = np.unique(y)

    from matplotlib.cm import get_cmap
    cmap = get_cmap("tab10")
    color = {c: cmap(i % 10) for i, c in enumerate(classes)}

    plt.figure(figsize=(7, 6))
    for c in classes:
        idx = np.where(y == c)[0]
        col = color[c]
        # sample up to n_traces indices for this class
        if len(idx) > n_traces:
            rng = np.random.default_rng(0)
            idx_sample = rng.choice(idx, size=n_traces, replace=False)
        else:
            idx_sample = idx
        # single-trial traces (same number per class)
        for i in idx_sample:
            plt.plot(Dcoord[i], Zcoord[i], color=col, alpha=0.25, lw=1.0)
        # class mean
        plt.plot(Dcoord[idx].mean(0), Zcoord[idx].mean(0), color=col, lw=3.0, label=f"class {c}")

    plt.axhline(0, color="k", lw=0.8, alpha=0.3)
    plt.axvline(0, color="k", lw=0.8, alpha=0.3)
    plt.xlabel("decision axis (d)")
    plt.ylabel("orthogonal PC (z)")
    plt.title(title)
    plt.legend(frameon=False)
    plt.tight_layout()
    plt.show()


def vector_field_from_trajs(Dcoord, Zcoord, dt=1.0, nbins=25, min_count=5,
                            extent=None, density_bg=True, title="Vector field in (d, z)"):
    """
    Build a nonparametric vector field from trajectories in (d,z).
    - Dcoord, Zcoord: arrays (N, T)
    - dt: timestep between consecutive columns in Dcoord/Zcoord
    - nbins: grid resolution for binning the field
    - min_count: minimum samples in a bin to draw an arrow
    - extent: ((dmin,dmax),(zmin,zmax)) overrides auto range
    - density_bg: whether to draw a background density image
    """
    N, T = Dcoord.shape
    # finite differences (centered on t -> t+1)
    d = Dcoord[:, :-1]; z = Zcoord[:, :-1]
    dd = (Dcoord[:, 1:] - Dcoord[:, :-1]) / dt
    dz = (Zcoord[:, 1:] - Zcoord[:, :-1]) / dt

    # flatten all points
    d_all = d.ravel(); z_all = z.ravel()
    dd_all = dd.ravel(); dz_all = dz.ravel()

    # grid
    if extent is None:
        pad = 1e-6
        dmin, dmax = np.percentile(d_all, [1, 99]); dpad = 0.05*(dmax-dmin+pad)
        zmin, zmax = np.percentile(z_all, [1, 99]); zpad = 0.05*(zmax-zmin+pad)
        dmin -= dpad; dmax += dpad; zmin -= zpad; zmax += zpad
    else:
        (dmin,dmax),(zmin,zmax) = extent

    di = np.clip(((d_all - dmin) / (dmax - dmin) * nbins).astype(int), 0, nbins-1)
    zi = np.clip(((z_all - zmin) / (zmax - zmin) * nbins).astype(int), 0, nbins-1)

    # accumulate sums per bin
    U = np.zeros((nbins, nbins), dtype=float)  # dd/dt
    V = np.zeros((nbins, nbins), dtype=float)  # dz/dt
    C = np.zeros((nbins, nbins), dtype=int)    # counts

    for i in range(d_all.size):
        U[zi[i], di[i]] += dd_all[i]
        V[zi[i], di[i]] += dz_all[i]
        C[zi[i], di[i]] += 1

    # average vectors
    mask = C >= min_count
    U[mask] /= C[mask]
    V[mask] /= C[mask]

    # grid centers for plotting
    d_edges = np.linspace(dmin, dmax, nbins+1)
    z_edges = np.linspace(zmin, zmax, nbins+1)
    d_cent = 0.5*(d_edges[:-1] + d_edges[1:])
    z_cent = 0.5*(z_edges[:-1] + z_edges[1:])
    Dg, Zg = np.meshgrid(d_cent, z_cent)

    plt.figure(figsize=(7,6))

    # optional density background
    if density_bg:
        H, _, _ = np.histogram2d(d_all, z_all, bins=[d_edges, z_edges])
        plt.imshow(H.T, origin="lower",
                   extent=[dmin, dmax, zmin, zmax],
                   aspect="auto", alpha=0.35)

    # quiver of averaged vectors
    Uplot = np.where(mask, U, 0.0)
    Vplot = np.where(mask, V, 0.0)
    # normalize arrow lengths a bit for visibility
    scale = np.nanpercentile(np.hypot(Uplot[mask], Vplot[mask]), 90) or 1.0
    plt.quiver(Dg[mask], Zg[mask], Uplot[mask]/scale, Vplot[mask]/scale,
               angles='xy', scale_units='xy', scale=1.0, width=0.003, alpha=0.9)

    plt.axhline(0, color='k', lw=0.8, alpha=0.3); plt.axvline(0, color='k', lw=0.8, alpha=0.3)
    plt.xlabel("S_1")
    plt.ylabel("S_2")
    plt.title(title)
    plt.tight_layout()
    plt.show()


In [ ]:
train_steps = np.arange(30)
act = train_ea[:, train_steps]
act = act.reshape(*act.shape[:-3], -1)

In [ ]:
X_train, y_train = act, train_decisions
print(X_train.shape, y_train.shape)

clf, d = train_linear_decoder(X_train, y_train)
_, R = project_to_d_and_residual(X_train, d)
z, _ = pca_in_d_perp(R, d)

In [ ]:
X_plot,  y_plot  = act, train_decisions
Dcoord, Zcoord = project_to_plane(X_plot[:, 2:], d, z)
vector_field_from_trajs(Dcoord, Zcoord, dt=1.0, nbins=20, min_count=10,
                        title="Empirical vector field in (S_1, S_2)")

In [ ]:
X_plot,  y_plot  = act, train_decisions
print(X_plot.shape, y_plot.shape)
plot_trajectories_in_plane(X_plot[:, :], y_plot, d, z, "Decision-plane trajectories", n_traces=5)

In [ ]:
X_plot,  y_plot  = act, train_decisions
print(X_plot.shape, y_plot.shape)
plot_trajectories_in_plane(X_plot[:, :], y_plot, d, z, "Decision-plane trajectories", n_traces=5)

In [ ]:
X_plot,  y_plot  = act, train_decisions
print(X_plot.shape, y_plot.shape)
plot_trajectories_in_plane(X_plot[:, :], y_plot, d, z, "Decision-plane trajectories", n_traces=5)

In [ ]:
decoder_args = {
    "hidden": 32,
    "task": "multiclass",
    "epochs": 10,
    "dropout": 0.1,
    "lr": 1e-4,
    "bs": 256
}

decoder_model, results = train_mlp_decoder(
    torch.tensor(act),
    decoder_train_labels,
    torch.tensor(test_act),
    decoder_test_labels,
    **decoder_args
)

In [ ]:
results

In [ ]:
# test multiclass or binary tasks
with torch.no_grad():
    logits = decoder_model(torch.tensor(act).to(device))      # [N, 8]
    pred = logits.argmax(1)
    
    wrong = 0
    for i in range(len(pred)):
        if pred[i] != decoder_train_labels[i]:
            wrong += 1
    print(f"Train acc: {1-(wrong/len(pred))}")

    logits = decoder_model(torch.tensor(test_act).to(device))      # [N, 8]
    pred = logits.argmax(1)
        
    wrong = 0
    for i in range(len(pred)):
        if pred[i] != decoder_test_labels[i]:
            wrong += 1
    print(f"Val acc: {1-(wrong/len(pred))}")

In [ ]:
# test regression taslk
with torch.no_grad():
    pred = decoder_model(torch.tensor(act).to(device)).cpu().numpy()[:, 0]
    print(f"Train error: {np.abs((pred - decoder_train_labels.numpy())).mean()}")

    pred = decoder_model(torch.tensor(test_act).to(device)).cpu().numpy()[:, 0]
    print(f"Val error: {np.abs((pred - decoder_test_labels.numpy())).mean()}")

In [ ]:
# X: [N, D]
start_idx = 0
end_idx = 30

#TRAIN
pca_acts = train_ea[:, start_idx:end_idx]
decisions = train_decisions

#TEST
# pca_acts = test_ea[:, start_idx:end_idx]
# decisions = test_decisions

num_samples = len(pca_acts)

pca_acts = pca_acts.reshape(*pca_acts.shape[:-3], -1)
pca_acts = pca_acts.reshape(-1, pca_acts.shape[-1])

# Process in batches to avoid OOM
batch_size = 1024
hidden_list = []
with torch.no_grad():
    for i in range(0, pca_acts.shape[0], batch_size):
        batch = pca_acts[i:i+batch_size]
        batch_hidden = hidden_from_mlp(decoder_model, torch.tensor(batch).to(device), apply_relu=True)
        hidden_list.append(batch_hidden.cpu())
pca_acts_hidden = torch.cat(hidden_list, dim=0).numpy()
pca_acts = pca_acts_hidden.reshape(num_samples, end_idx-start_idx, decoder_args["hidden"])

## Do the PCA

In [ ]:
p = PCTrajectoryPlotter(max_components=3)

# Train set (what you used to pass to plot_PC_trajectories)
p.fit(pca_acts, decisions, balance_for_fit=False, collapse_to_class_means=False)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def plot_vector_field_2d(trajs, dt=1.0, nbins=25, min_count=5,
                         density_bg=True, cmap_bg="Greys",
                         title="Vector field from trajectories",
                         vertical_aspect=1.3,  # >1 makes plot “taller” in data units (y per x)
                         fig_width=5.0):
    """
    trajs: (N,T,2)
    vertical_aspect: y/x data-unit ratio (1.0 = square units; 1.3 makes y look taller)
    fig_width: inches; height is set so the axes honor vertical_aspect without whitespace
    """
    trajs = np.asarray(trajs)
    if trajs.ndim != 3 or trajs.shape[2] != 2:
        raise ValueError("trajs must be (N,T,2)")
    N, T, _ = trajs.shape

    # data + velocities
    x, y = trajs[..., 0], trajs[..., 1]
    dx, dy = np.diff(x, axis=1) / dt, np.diff(y, axis=1) / dt
    x_mid, y_mid = x[:, :-1], y[:, :-1]

    # flatten for binning
    x_all, y_all = x_mid.ravel(), y_mid.ravel()
    dx_all, dy_all = dx.ravel(), dy.ravel()

    # tight but padded limits (trim vertical whitespace)
    x_min, x_max = np.percentile(x_all, [2, 98])
    y_min, y_max = np.percentile(y_all, [2, 98])
    pad_x, pad_y = 0.06*(x_max-x_min + 1e-12), 0.06*(y_max-y_min + 1e-12)
    x_min, x_max = x_min - pad_x, x_max + pad_x
    y_min, y_max = y_min - pad_y, y_max + pad_y

    # binning
    xi = np.clip(((x_all - x_min) / (x_max - x_min) * nbins).astype(int), 0, nbins-1)
    yi = np.clip(((y_all - y_min) / (y_max - y_min) * nbins).astype(int), 0, nbins-1)

    U = np.zeros((nbins, nbins))
    V = np.zeros((nbins, nbins))
    C = np.zeros((nbins, nbins), dtype=int)
    for i in range(x_all.size):
        U[yi[i], xi[i]] += dx_all[i]
        V[yi[i], xi[i]] += dy_all[i]
        C[yi[i], xi[i]] += 1

    mask = C >= min_count
    U[mask] /= C[mask]
    V[mask] /= C[mask]

    # grid centers
    x_edges = np.linspace(x_min, x_max, nbins+1)
    y_edges = np.linspace(y_min, y_max, nbins+1)
    x_cent = 0.5*(x_edges[:-1] + x_edges[1:])
    y_cent = 0.5*(y_edges[:-1] + y_edges[1:])
    Xg, Yg = np.meshgrid(x_cent, y_cent)

    # compute figure height so axes can honor the requested data aspect without big margins
    data_ratio = (y_max - y_min) / (x_max - x_min)  # current y/x span
    wanted_ratio = vertical_aspect                   # y/x units on the axes
    # choose a height that roughly accommodates the desired axes ratio
    fig_height = fig_width * (wanted_ratio / 1.0)

    fig, ax = plt.subplots(figsize=(fig_width, fig_height))

    if density_bg:
        H, _, _ = np.histogram2d(x_all, y_all, bins=[x_edges, y_edges])
        ax.imshow(H.T, origin="lower",
                  extent=[x_min, x_max, y_min, y_max],
                  interpolation="bilinear", cmap=cmap_bg, alpha=0.25)

    # arrows
    Uplot, Vplot = np.where(mask, U, 0.0), np.where(mask, V, 0.0)
    mag = np.hypot(Uplot[mask], Vplot[mask])
    scale = np.nanpercentile(mag, 95) or 1.0
    ax.quiver(Xg[mask], Yg[mask],
              Uplot[mask]/scale, Vplot[mask]/scale,
              angles="xy", scale_units="xy", scale=1.0,
              width=0.002, headwidth=2.5, headlength=3.5,
              alpha=0.9, color="tab:blue")

    # axes + styling
    ax.axhline(0, color='k', lw=0.6, alpha=0.3)
    ax.axvline(0, color='k', lw=0.6, alpha=0.3)
    ax.set_xlim(x_min, x_max)
    ax.set_ylim(y_min, y_max)
    ax.set_xlabel("PC 1", fontsize=8); ax.set_ylabel("PC 2", fontsize=8); ax.set_title(title, fontsize=10)
    ax.spines[['top','right']].set_visible(False)

    # **this is the key line**: set the *data* aspect (y per x unit)
    ax.set_aspect(wanted_ratio, adjustable='box')

    plt.tight_layout()
    plt.show()


In [ ]:
pca_proj = p.plot(
    pca_acts[:, :], decisions,
    overlay_examples_k=3,
    overlay_mode="closest",     # 'closest' | 'farthest' | 'random'
    overlay_classes=None,       # None => all classes
    plot_all_endpoints=False,    # scatter ENDPOINTS for all trials
    per_class_mean=True,
    plot_trajectory=False,
    title="Coherence = 0.25",
    alpha=0.75,
    collapse_to_class_means=False,
    extra_point_timestep=39,
    n_components=2,
    return_data=True
)

In [ ]:
plot_vector_field_2d(pca_proj[:, 2:], min_count=5, nbins=30, title="Empirical Vector Field", vertical_aspect=2)

In [ ]:
p.plot(
    pca_acts[:, :], decisions,
    overlay_examples_k=6,
    overlay_mode="closest",     # 'closest' | 'farthest' | 'random'
    overlay_classes=None,       # None => all classes
    plot_all_endpoints=True,    # scatter ENDPOINTS for all trials
    per_class_mean=True,
    plot_trajectory=False,
    title="Train trajectories",
    alpha=0.75,
    collapse_to_class_means=False,
    extra_point_timestep=39,
    n_components=2
)

In [ ]:
pca_acts.shape

In [ ]:
p.plot(
    pca_acts[:, :], decisions,
    overlay_examples_k=6,
    overlay_mode="closest",     # 'closest' | 'farthest' | 'random'
    overlay_classes=None,       # None => all classes
    plot_all_endpoints=True,    # scatter ENDPOINTS for all trials
    per_class_mean=True,
    plot_trajectory=False,
    title="Train trajectories",
    alpha=0.75,
    collapse_to_class_means=False,
    extra_point_timestep=39,
    n_components=2
)

In [ ]:
def rotate_points(x, y, theta=np.pi/4):
    """Rotate (x,y) by -theta (clockwise)."""
    R = np.array([[np.cos(theta), -np.sin(theta)],
                  [np.sin(theta),  np.cos(theta)]])
    xy = np.vstack([x, y])  # shape (2, N)
    xy_rot = R @ xy
    return xy_rot[0], xy_rot[1]

def reflect_over_y_eq_ax(x, y, a=1.0):
    """
    Reflect points (x,y) across the line y = a*x.
    Works with scalars or arrays.
    """
    R = np.array([[1 - a**2,  2*a],
                  [2*a,       a**2 - 1]]) / (1 + a**2)
    xy = np.vstack([x, y])   # shape (2, N)
    xy_ref = R @ xy
    return xy_ref[0], xy_ref[1]

def plot_integrated_direction(x, y, ax=None, label=0):
    if ax is None:
        fig, ax = plt.subplots(figsize=(6, 4))

    y = -y
    x = -x

    x, y = rotate_points(x, y, theta=np.pi/4)

    # Reflect the trajectory
    # x, y = reflect_over_y_eq_ax(x, y, a=a)

    points = np.array([x, y]).T.reshape(-1, 1, 2)
    segments = np.concatenate([points[:-1], points[1:]], axis=1)
    timesteps = np.arange(num_steps)

    lc = LineCollection(segments, cmap='viridis',
                        norm=plt.Normalize(timesteps[0], timesteps[-1]))
    lc.set_array(timesteps[:-1])  # Use timesteps for segments, not points
    lc.set_linewidth(3)

    line = ax.add_collection(lc)
    ax.scatter(x[-1], y[-1], label=label, s=20)

    # Set x and y limits with equal scaling
    x_min, x_max = x.min() - 1, x.max() + 1
    y_min, y_max = y.min() - 1, y.max() + 1
    # Find the center and the max range to make axes equal
    x_center = (x_min + x_max) / 2
    y_center = (y_min + y_max) / 2
    half_range = max(x_max - x_min, y_max - y_min) / 2
    ax.set_xlim(x_center - half_range, x_center + half_range)
    ax.set_ylim(y_center - half_range, y_center + half_range)
    ax.set_aspect('equal', adjustable='box')
    ax.legend()


In [ ]:
# Subset-aligned arrays (same order/length as integrated_directions)
labels_subset = train_labels[subset_indices]
pca_acts_subset = pca_acts[subset_indices]
decisions_subset = decisions[subset_indices]

unique_labels = np.unique(labels_subset)
n_labels = len(unique_labels)

# For each label, get positions *within the subset* (0..num_intg_examples-1)
label_to_subset_idxs = {lbl: np.where(labels_subset == lbl)[0] for lbl in unique_labels}

# Rows = up to the largest label count (cap at 10 as before)
max_rows = min(max(len(v) for v in label_to_subset_idxs.values()), 17)

fig, axes = plt.subplots(max_rows, 2, figsize=(12, 4 * max_rows))
if max_rows == 1:
    axes = np.expand_dims(axes, 0)  # keep 2D indexing

for row in range(max_rows):
    int_ax = axes[row, 0]
    pca_ax = axes[row, 1]

    subset_pos_list = []
    labels_in_row = []
    for lbl in unique_labels:
        idxs = label_to_subset_idxs[lbl]
        if row < len(idxs):
            subset_pos_list.append(idxs[row])
            labels_in_row.append(lbl)

    if not subset_pos_list:
        int_ax.axis("off")
        pca_ax.axis("off")
        continue

    # ---- LEFT: overlay all integrated trajectories for this row ----
    # Draw all curves first (your helper sets its own limits; we will override).
    max_abs = 0.0  # CENTER AT ORIGIN: track max radius across all overlays

    for pos in subset_pos_list:
        x = integrated_directions[pos, 2:, 1]
        y = integrated_directions[pos, 2:, 0]

        plt.sca(int_ax)
        plot_integrated_direction(x, y, ax=int_ax, label=labels_subset[pos])

        # match helper's y inversion for bound calc
        y_plotted = -y
        # CENTER AT ORIGIN: track symmetric extent
        max_abs = max(max_abs, np.max(np.abs(x)), np.max(np.abs(y_plotted)))

    # CENTER AT ORIGIN: set symmetric limits that fit everything (+ a small margin)
    margin = 1.05
    R = max_abs * margin if max_abs > 0 else 1.0
    int_ax.set_xlim(-R, R)
    int_ax.set_ylim(-R, R)
    int_ax.set_aspect('equal', adjustable='box')

    # Show axes crossing at (0,0)
    int_ax.axhline(0, linewidth=1, alpha=0.6)
    int_ax.axvline(0, linewidth=1, alpha=0.6)

    int_ax.set_title(f"Integrated directions | Row {row}")
    int_ax.set_xlabel("X")
    int_ax.set_ylabel("Y")

    # ---- RIGHT: PCA on stacked examples for this row ----
    X_row = pca_acts_subset[subset_pos_list, 2:]
    D_row = decisions_subset[subset_pos_list]

    pca_ax.set_aspect('equal', adjustable='datalim')
    pca_ax.set_box_aspect(1)
    p.plot(
        X_row, D_row,
        overlay_examples_k=6,
        overlay_mode="farthest",
        overlay_classes=None,
        plot_all_endpoints=True,
        per_class_mean=True,
        plot_trajectory=False,
        title="PCA",
        alpha=0.75,
        collapse_to_class_means=False,
        extra_point_timestep=39,
        n_components=2,
        ax=pca_ax,
        label_segments_with_timesteps=False,
        show=False
    )

fig.suptitle("Per-row overlays: Integrated (left, centered at 0,0) & PCA (right)", fontsize=18)
plt.tight_layout(rect=[0, 0, 1, 0.97])
plt.show()

In [ ]:
labels_subset = train_labels[subset_indices]
pca_acts_subset = pca_acts[subset_indices]
decisions_subset = decisions[subset_indices]

unique_labels = np.unique(labels_subset)
n_labels = len(unique_labels)

# find the maximum number of examples in any label
max_rows = min(max(np.sum(labels_subset == l) for l in unique_labels), 10)

fig, axes = plt.subplots(
    max_rows, 2 * n_labels,
    figsize=(6 * n_labels, 4 * max_rows)
)

if max_rows == 1:
    axes = np.expand_dims(axes, 0)  # make axes 2D for consistent indexing

for col, label in enumerate(unique_labels):
    idxs = np.where(labels_subset == label)[0]

    for row, i in enumerate(idxs):
        int_ax = axes[row, 2*col]      # left cell for this label
        pca_ax = axes[row, 2*col + 1]  # right cell for this label

        # --- Integrated direction ---
        x = integrated_directions[i, 2:, 1]
        y = integrated_directions[i, 2:, 0]
        plt.sca(int_ax)  # force plotting to this axis
        plot_integrated_direction(x, y, ax=int_ax)
        int_ax.set_title(f"Label {label} | Ex {i}")
        int_ax.set_xlabel("X")
        int_ax.set_ylabel("Y")
        int_ax.set_box_aspect(1)

        # --- PCA plot ---
        pca_ax.set_aspect('equal', adjustable='datalim')
        pca_ax.set_box_aspect(1)
        p.plot(
            pca_acts_subset[i:i+1, 2:], decisions_subset[i:i+1],
            overlay_examples_k=6,
            overlay_mode="farthest",
            overlay_classes=None,
            plot_all_endpoints=True,
            per_class_mean=True,
            plot_trajectory=False,
            title=f"PCA (Label {label}, Ex {i})",
            alpha=0.75,
            collapse_to_class_means=False,
            extra_point_timestep=39,
            n_components=2,
            ax=pca_ax,
            label_segments_with_timesteps=False,
            show=False
        )
        if row == max_rows-1:
            break

    # Hide unused rows for this label if some labels have fewer examples
    for row in range(len(idxs), max_rows):
        axes[row, 2*col].axis("off")
        axes[row, 2*col+1].axis("off")

fig.suptitle("All Labels and Examples", fontsize=20)
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()

In [ ]:
unique_labels = np.unique(train_labels)

for label in unique_labels:
    idxs = np.where(train_labels[:num_intg_examples] == label)[0]

    fig, axes = plt.subplots(len(idxs), 2, figsize=(12, 4 * len(idxs)))
    if len(idxs) == 1:
        # Normalize shape so axes[row, col] indexing works
        axes = np.array([axes])

    for row, i in enumerate(idxs):
        int_ax = axes[row, 0]
        pca_ax = axes[row, 1]

        # --- Integrated direction ---
        x = integrated_directions[i, 2:, 1]
        y = integrated_directions[i, 2:, 0]

        # ensure plotting goes to the correct axes even if the helper uses plt.plot
        plt.sca(int_ax)
        try:
            # if your helper *does* accept ax, pass it too (harmless if ignored)
            plot_integrated_direction(x, y, ax=int_ax)
        except TypeError:
            # fallback if it doesn't accept ax
            plot_integrated_direction(x, y)

        int_ax.set_title(f"Integrated directions (label {label}, ex {i})")
        int_ax.set_xlabel("Direction X (rotated)")
        int_ax.set_ylabel("Direction Y (rotated)")
        int_ax.set_box_aspect(1)

        # --- PCA plot ---
        pca_ax.set_aspect('equal', adjustable='datalim')
        pca_ax.set_box_aspect(1)
        p.plot(
            pca_acts[i:i+1, 2:], decisions[i:i+1],
            overlay_examples_k=6,
            overlay_mode="farthest",
            overlay_classes=None,
            plot_all_endpoints=True,
            per_class_mean=True,
            plot_trajectory=False,
            title=f"Train trajectories (label {label}, ex {i})",
            alpha=0.75,
            collapse_to_class_means=False,
            extra_point_timestep=39,
            n_components=2,
            ax=pca_ax,
            label_segments_with_timesteps=False,
            show=False
        )

    fig.suptitle(f"All examples for label {label}", fontsize=16)
    plt.tight_layout(rect=[0, 0, 1, 0.97])
    plt.show()


# try no decision space and cor=0.25, 0.4

In [ ]:
p.plot(
    pca_acts[:, 1:], decisions,
    overlay_examples_k=6,
    overlay_mode="farthest",     # 'closest' | 'farthest' | 'random'
    overlay_classes=None,       # None => all classes
    plot_all_endpoints=True,    # scatter ENDPOINTS for all trials
    per_class_mean=True,
    plot_trajectory=False,
    title="Train trajectories",
    alpha=0.75,
    collapse_to_class_means=False,
    extra_point_timestep=39,
    n_components=2
)

In [ ]:
for i in range(num_intg_examples):

    fig, (int_ax, pca_ax) = plt.subplots(1, 2, figsize=(12, 4))
    label = train_labels[i]

    # Original trajectory
    x = integrated_directions[i, :, 1]
    y = integrated_directions[i, :, 0]

    # Plot integrated direction on the first axis
    plot_integrated_direction(x, y, ax=int_ax)
    plt.sca(int_ax)  # Set current axis to int_ax for any further plotting
    int_ax.set_title(f"Integrated directions, label: {train_labels[i]}")
    int_ax.set_xlabel("Direction X (rotated)")
    int_ax.set_ylabel("Direction Y (rotated)")
    int_ax.set_box_aspect(1)
    
    # set pca_ax x and y scales to be the same as each other
    pca_ax.set_aspect('equal', adjustable='datalim')  # equal data units; limits can be whatever
    pca_ax.set_box_aspect(1)      

    # Plot PCA on the second axis
    p.plot(
        pca_acts[i:i+1, :], decisions[i:i+1],
        overlay_examples_k=6,
        overlay_mode="farthest",     # 'closest' | 'farthest' | 'random'
        overlay_classes=None,       # None => all classes
        plot_all_endpoints=True,    # scatter ENDPOINTS for all trials
        per_class_mean=True,
        plot_trajectory=False,
        title="Train trajectories",
        alpha=0.75,
        collapse_to_class_means=False,
        extra_point_timestep=39,
        n_components=2,
        ax=pca_ax,
        label_segments_with_timesteps=False
    )

    plt.tight_layout()
    plt.show()

In [ ]:
# Explore this more. maybe try plotting a higher correlation. make sure to check axis are somewhat aligned like they are here.

# Run exactly this but with more samples and look at directions individually. And plot all in one plot.

for i in range(num_intg_examples):

    fig, (int_ax, pca_ax) = plt.subplots(1, 2, figsize=(12, 4))

    # Plot integrated direction on the first axis
    plot_integrated_direction(i, ax=int_ax)
    plt.sca(int_ax)  # Set current axis to int_ax for any further plotting
    int_ax.set_title(f"Integrated directions (rotated) for sample {i}, label: {train_labels[i]}")
    int_ax.set_xlabel("Direction X (rotated)")
    int_ax.set_ylabel("Direction Y (rotated)")
    int_ax.set_box_aspect(1)
    
    # set pca_ax x and y scales to be the same as each other
    pca_ax.set_aspect('equal', adjustable='datalim')  # equal data units; limits can be whatever
    pca_ax.set_box_aspect(1)      

    # Plot PCA on the second axis
    p.plot(
        pca_acts[i:i+1, :], decisions[i:i+1],
        overlay_examples_k=6,
        overlay_mode="farthest",     # 'closest' | 'farthest' | 'random'
        overlay_classes=None,       # None => all classes
        plot_all_endpoints=True,    # scatter ENDPOINTS for all trials
        per_class_mean=True,
        plot_trajectory=False,
        title="Train trajectories",
        alpha=0.75,
        collapse_to_class_means=False,
        extra_point_timestep=39,
        n_components=2,
        ax=pca_ax,
        label_segments_with_timesteps=False
    )

    plt.tight_layout()
    plt.show()

In [ ]:
# Explore this more. maybe try plotting a higher correlation. make sure to check axis are somewhat aligned like they are here.

for i in range(num_intg_examples):
    fig, (int_ax, pca_ax) = plt.subplots(1, 2, figsize=(12, 4))

    # Plot integrated direction on the first axis
    plot_integrated_direction(i, ax=int_ax)
    plt.sca(int_ax)  # Set current axis to int_ax for any further plotting
    int_ax.set_title(f"Integrated directions (rotated) for sample {i}, label: {train_labels[i]}")
    int_ax.set_xlabel("Direction X (rotated)")
    int_ax.set_ylabel("Direction Y (rotated)")
    int_ax.set_box_aspect(1)
    
    # set pca_ax x and y scales to be the same as each other
    pca_ax.set_aspect('equal', adjustable='datalim')  # equal data units; limits can be whatever
    pca_ax.set_box_aspect(1)      

    # Plot PCA on the second axis
    p.plot(
        pca_acts[i:i+1, :], decisions[i:i+1],
        overlay_examples_k=6,
        overlay_mode="farthest",     # 'closest' | 'farthest' | 'random'
        overlay_classes=None,       # None => all classes
        plot_all_endpoints=True,    # scatter ENDPOINTS for all trials
        per_class_mean=True,
        plot_trajectory=False,
        title="Train trajectories",
        alpha=0.75,
        collapse_to_class_means=False,
        extra_point_timestep=39,
        n_components=2,
        ax=pca_ax,
        label_segments_with_timesteps=False
    )

    plt.tight_layout()
    plt.show()

In [ ]:
p.plot(
    pca_acts, decisions,
    overlay_examples_k=6,
    overlay_mode="farthest",     # 'closest' | 'farthest' | 'random'
    overlay_classes=None,       # None => all classes
    plot_all_endpoints=False,    # scatter ENDPOINTS for all trials
    per_class_mean=True,
    plot_trajectory=False,
    title="Train trajectories",
    alpha=0.75,
    collapse_to_class_means=False,
    extra_point_timestep=39,
    n_components=2
)

p.plot(
    pca_acts, decisions,
    overlay_examples_k=6,
    overlay_mode="farthest",     # 'closest' | 'farthest' | 'random'
    overlay_classes=None,       # None => all classes
    plot_all_endpoints=True,    # scatter ENDPOINTS for all trials
    per_class_mean=True,
    plot_trajectory=False,
    title="Train trajectories",
    alpha=0.75,
    collapse_to_class_means=False,
    extra_point_timestep=39,
    n_components=2
)

In [ ]:
p.plot(
    pca_acts[:, 3:], decisions,
    overlay_examples_k=6,
    overlay_mode="farthest",     # 'closest' | 'farthest' | 'random'
    overlay_classes=None,       # None => all classes
    plot_all_endpoints=False,    # scatter ENDPOINTS for all trials
    per_class_mean=True,
    plot_trajectory=False,
    title="Train trajectories",
    alpha=0.75,
    collapse_to_class_means=False,
    extra_point_timestep=39,
    n_components=2
)

In [ ]:
p.plot(
    pca_acts[:, 3:], decisions,
    overlay_examples_k=6,
    overlay_mode="farthest",     # 'closest' | 'farthest' | 'random'
    overlay_classes=None,       # None => all classes
    plot_all_endpoints=False,    # scatter ENDPOINTS for all trials
    per_class_mean=True,
    plot_trajectory=False,
    title="Train trajectories",
    alpha=0.75,
    collapse_to_class_means=False,
    extra_point_timestep=39,
    n_components=2
)

p.plot(
    pca_acts[:, 3:], decisions,
    overlay_examples_k=6,
    overlay_mode="farthest",     # 'closest' | 'farthest' | 'random'
    overlay_classes=None,       # None => all classes
    plot_all_endpoints=True,    # scatter ENDPOINTS for all trials
    per_class_mean=True,
    plot_trajectory=False,
    title="Train trajectories",
    alpha=0.75,
    collapse_to_class_means=False,
    extra_point_timestep=39,
    n_components=2
)

In [ ]:
p.plot(
    pca_acts, decisions,
    overlay_examples_k=6,
    overlay_mode="farthest",     # 'closest' | 'farthest' | 'random'
    overlay_classes=None,       # None => all classes
    plot_all_endpoints=False,    # scatter ENDPOINTS for all trials
    per_class_mean=True,
    plot_trajectory=False,
    title="Train trajectories",
    alpha=0.75,
    collapse_to_class_means=False,
    extra_point_timestep=39,
    n_components=2
)

p.plot(
    pca_acts, decisions,
    overlay_examples_k=6,
    overlay_mode="farthest",     # 'closest' | 'farthest' | 'random'
    overlay_classes=None,       # None => all classes
    plot_all_endpoints=True,    # scatter ENDPOINTS for all trials
    per_class_mean=True,
    plot_trajectory=False,
    title="Train trajectories",
    alpha=0.75,
    collapse_to_class_means=False,
    extra_point_timestep=39,
    n_components=2
)

In [ ]:
p.plot(
    pca_acts, decisions,
    overlay_examples_k=6,
    overlay_mode="farthest",     # 'closest' | 'farthest' | 'random'
    overlay_classes=None,       # None => all classes
    plot_all_endpoints=False,    # scatter ENDPOINTS for all trials
    per_class_mean=True,
    plot_trajectory=False,
    title="Train trajectories",
    alpha=0.75,
    collapse_to_class_means=False,
    extra_point_timestep=39,
    n_components=2
)

p.plot(
    pca_acts, decisions,
    overlay_examples_k=6,
    overlay_mode="farthest",     # 'closest' | 'farthest' | 'random'
    overlay_classes=None,       # None => all classes
    plot_all_endpoints=True,    # scatter ENDPOINTS for all trials
    per_class_mean=True,
    plot_trajectory=False,
    title="Train trajectories",
    alpha=0.75,
    collapse_to_class_means=False,
    extra_point_timestep=39,
    n_components=2
)

In [ ]:
early_late_tensor

In [ ]:
# Same behavior as before: overlay k exemplars per class (chosen by ENDPOINT),
# draw class means, and optionally show all endpoints.

p.plot(
    pca_acts, decisions_tensor,
    overlay_examples_k=6,
    overlay_mode="random",     # 'closest' | 'farthest' | 'random'
    overlay_classes=None,       # None => all classes
    plot_all_endpoints=False,    # scatter ENDPOINTS for all trials
    per_class_mean=False,
    plot_trajectory=False,
    title="Train trajectories",
    alpha=0.75,
    collapse_to_class_means=False,
    extra_point_timestep=39,
    n_components=2
)

p.plot(
    pca_acts, labels_tensor,
    overlay_examples_k=6,
    overlay_mode="random",     # 'closest' | 'farthest' | 'random'
    overlay_classes=None,       # None => all classes
    plot_all_endpoints=False,    # scatter ENDPOINTS for all trials
    per_class_mean=False,
    plot_trajectory=False,
    title="Train trajectories",
    alpha=0.75,
    collapse_to_class_means=False,
    extra_point_timestep=39,
    n_components=2
)

early_late_tensor = early_late_tensor.clone()
early_late_tensor[early_late_tensor == 0] = 2
early_late_tensor[early_late_tensor == 1] = 4

p.plot(
    pca_acts, early_late_tensor,
    overlay_examples_k=6,
    overlay_mode="random",     # 'closest' | 'farthest' | 'random'
    overlay_classes=None,       # None => all classes
    plot_all_endpoints=False,    # scatter ENDPOINTS for all trials
    per_class_mean=False,
    plot_trajectory=False,
    title="Train trajectories",
    alpha=0.75,
    collapse_to_class_means=False,
    extra_point_timestep=39,
    n_components=2
)

#These are potentially interesting, checking decision decoding

In [ ]:
p.plot(
    pca_acts, decisions,
    overlay_examples_k=6,
    overlay_mode="random",     # 'closest' | 'farthest' | 'random'
    overlay_classes=None,       # None => all classes
    plot_all_endpoints=False,    # scatter ENDPOINTS for all trials
    per_class_mean=False,
    plot_trajectory=False,
    title="Train trajectories",
    alpha=0.75,
    collapse_to_class_means=False,
    extra_point_timestep=39,
    n_components=3
)

In [ ]:
# p = PCTrajectoryPlotter(n_components=2, random_state=0)

# # Train set (what you used to pass to plot_PC_trajectories)
# p.fit(pca_acts, decisions, balance_for_fit=False, collapse_to_class_means=True)

# Same behavior as before: overlay k exemplars per class (chosen by ENDPOINT),
# draw class means, and optionally show all endpoints.
p.plot(
    pca_acts, decisions,
    overlay_examples_k=10,
    overlay_mode="random",     # 'closest' | 'farthest' | 'random'
    overlay_classes=None,       # None => all classes
    plot_all_endpoints=True,    # scatter ENDPOINTS for all trials
    per_class_mean=False,
    plot_trajectory=False,
    title="Train trajectories",
    alpha=0.75,
    collapse_to_class_means=False
)

In [ ]:
u = UMAPTrajectoryPlotter(n_components=3, n_neighbors=25, min_dist=0.05, random_state=0)

# Fit on train (trial-level embedding)
u.fit(pca_acts, decisions, balance_for_fit=False)

# Plot train
u.plot(
    act, train_decisions,
    overlay_examples_k=5,
    overlay_mode="closest",
    overlay_classes=None,        # or [0, 2, ...]
    plot_all_endpoints=True,
    per_class_mean=True,
    title="UMAP: Train trajectories"
)


In [ ]:
p1 = PCTrajectoryPlotter(n_components=3, random_state=0)

# act = train_oa[:num_train_samples, start_idx:end_idx]
# act = act.reshape(*act.shape[:-3], -1)

# Train set (what you used to pass to plot_PC_trajectories)
p1.fit(out, train_decisions, balance_for_fit=False, collapse_to_class_means=True)

# Same behavior as before: overlay k exemplars per class (chosen by ENDPOINT),
# draw class means, and optionally show all endpoints.
p1.plot(
    out, train_decisions,
    overlay_examples_k=0,
    overlay_mode="closest",     # 'closest' | 'farthest' | 'random'
    overlay_classes=None,       # None => all classes
    plot_all_endpoints=True,    # scatter ENDPOINTS for all trials
    per_class_mean=True,
    plot_trajectory=False,
    title="Train trajectories",
    alpha=0.75,
    collapse_to_class_means=False
)

In [ ]:
p = PCTrajectoryPlotter(n_components=3, random_state=0)

# act = train_oa[:num_train_samples, start_idx:end_idx]
# act = act.reshape(*act.shape[:-3], -1)

# Train set (what you used to pass to plot_PC_trajectories)
p.fit(out, train_decisions, balance_for_fit=False, collapse_to_class_means=True)

# Same behavior as before: overlay k exemplars per class (chosen by ENDPOINT),
# draw class means, and optionally show all endpoints.
p.plot(
    out, train_decisions,
    overlay_examples_k=1,
    overlay_mode="closest",     # 'closest' | 'farthest' | 'random'
    overlay_classes=None,       # None => all classes
    plot_all_endpoints=True,    # scatter ENDPOINTS for all trials
    per_class_mean=False,
    plot_trajectory=False,
    title="Train trajectories",
    alpha=0.75,
    collapse_to_class_means=False
)

In [ ]:
p = PCTrajectoryPlotter(n_components=3, random_state=0)

# act = train_oa[:num_train_samples, start_idx:end_idx]
# act = act.reshape(*act.shape[:-3], -1)

# Train set (what you used to pass to plot_PC_trajectories)
p.fit(out, train_decisions, balance_for_fit=False, collapse_to_class_means=True)

# Same behavior as before: overlay k exemplars per class (chosen by ENDPOINT),
# draw class means, and optionally show all endpoints.
p.plot(
    out, train_decisions,
    overlay_examples_k=1,
    overlay_mode="closest",     # 'closest' | 'farthest' | 'random'
    overlay_classes=[],       # None => all classes
    plot_all_endpoints=True,    # scatter ENDPOINTS for all trials
    per_class_mean=True,
    plot_trajectory=False,
    title="Train trajectories",
    alpha=0.75,
    collapse_to_class_means=False
)

In [ ]:
# p3 = PCTrajectoryPlotter(n_components=3, random_state=0)
p2 = PCTrajectoryPlotter(n_components=2, random_state=0)

act = train_ea[:num_train_samples, start_idx:end_idx]
act = act.reshape(*act.shape[:-3], -1)

# Train set (what you used to pass to plot_PC_trajectories)
p2.fit(act, train_decisions, balance_for_fit=False, collapse_to_class_means=True)

# Same behavior as before: overlay k exemplars per class (chosen by ENDPOINT),
# draw class means, and optionally show all endpoints.
p2.plot(
    act, train_decisions,
    overlay_examples_k=1,
    overlay_mode="closest",     # 'closest' | 'farthest' | 'random'
    overlay_classes=[0, 4],       # None => all classes
    plot_all_endpoints=True,    # scatter ENDPOINTS for all trials
    per_class_mean=False,
    plot_trajectory=False,
    title="Train trajectories",
    alpha=0.75,
    collapse_to_class_means=False
)

In [ ]:
p = PCTrajectoryPlotter(n_components=3, random_state=0)

act = train_ea[:num_samples, start_idx:end_idx]
act = act.reshape(*act.shape[:-3], -1)

# Train set (what you used to pass to plot_PC_trajectories)
p.fit(act, train_decisions, balance_for_fit=False, collapse_to_class_means=True)

# Same behavior as before: overlay k exemplars per class (chosen by ENDPOINT),
# draw class means, and optionally show all endpoints.
p.plot(
    act, train_decisions,
    overlay_examples_k=1,
    overlay_mode="closest",     # 'closest' | 'farthest' | 'random'
    overlay_classes=[0, 4],       # None => all classes
    plot_all_endpoints=True,    # scatter ENDPOINTS for all trials
    per_class_mean=False,
    plot_trajectory=True,
    title="Train trajectories",
    alpha=0.5,
    collapse_to_class_means=True
)

In [ ]:
# p = PCTrajectoryPlotter(n_components=2, random_state=0)

start_idx = 0
end_idx = 20
act = train_ea[:, start_idx:end_idx]
act = act.reshape(*act.shape[:-3], -1)

# Train set (what you used to pass to plot_PC_trajectories)
# p.fit(act, train_decisions, balance_for_fit=False, collapse_to_class_means=True)

# Same behavior as before: overlay k exemplars per class (chosen by ENDPOINT),
# draw class means, and optionally show all endpoints.
p.plot(
    act, train_decisions,
    overlay_examples_k=5,
    overlay_mode="closest",     # 'closest' | 'farthest' | 'random'
    overlay_classes=None,       # None => all classes
    plot_all_endpoints=False,    # scatter ENDPOINTS for all trials
    per_class_mean=False,
    plot_trajectory=True,
    title="Train trajectories",
    alpha=1,
    collapse_to_class_means=False,
    balance=False
)

In [ ]:
act.shape

In [ ]:
p = PCTrajectoryPlotter(n_components=2, random_state=0)

start_idx = 0
end_idx = 20
act = train_ea[:, start_idx:end_idx]
act = act.reshape(*act.shape[:-3], -1)

# Train set (what you used to pass to plot_PC_trajectories)
p.fit(act, train_decisions, balance_for_fit=False, collapse_to_class_means=True)

# Same behavior as before: overlay k exemplars per class (chosen by ENDPOINT),
# draw class means, and optionally show all endpoints.
p.plot(
    act, train_decisions,
    overlay_examples_k=5,
    overlay_mode="closest",     # 'closest' | 'farthest' | 'random'
    overlay_classes=None,       # None => all classes
    plot_all_endpoints=False,    # scatter ENDPOINTS for all trials
    per_class_mean=False,
    plot_trajectory=True,
    title="Train trajectories",
    alpha=0.5,
    collapse_to_class_means=True,
    balance=False
)

p.plot(
    act, train_decisions,
    overlay_examples_k=5,
    overlay_mode="closest",     # 'closest' | 'farthest' | 'random'
    overlay_classes=None,       # None => all classes
    plot_all_endpoints=False,    # scatter ENDPOINTS for all trials
    per_class_mean=True,
    plot_trajectory=True,
    title="Train trajectories",
    alpha=0.5,
    collapse_to_class_means=False,
    balance=False
)

In [ ]:
num_samples = 1000
# act = train_oa[:num_samples, 19, :].reshape(num_samples, -1)
# plot_PCs(act, train_decisions[:num_samples], n_components=3)

accuracies = []
for n_step in tqdm(range(39,40)):
    act = train_ea[:num_samples, n_step, :].reshape(num_samples, -1)
    test_act = train_ea[:num_samples, n_step, :].reshape(num_samples, -1)
    lr, accuracy = decision_decoder(act, train_decisions[:num_samples], test=(test_act, train_decisions[:num_samples]))
    accuracies.append(accuracy)

In [ ]:
# DOTS
for train_activations, test_activations in zip([train_oa, train_ea, train_ia], [test_oa, test_ea, test_ia]):
    act = activations[:num_samples, 10:20].reshape(num_samples*10, -1)
    repeated_decisions = np.repeat(train_decisions[:num_samples], 10)
    test_act = test_activations[:num_samples, 10:20].reshape(num_samples*10, -1)
    test_repeated_decisions = np.repeat(test_decisions[:num_samples], 10)
    lr, accuracy = decision_decoder(act, repeated_decisions, test=(test_act, test_repeated_decisions))
    print(accuracy)

In [ ]:
accuracies

In [ ]:
plt.plot(accuracies)

In [ ]:
plt.plot(accuracies)